# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patheffects as path_effects
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
import plotly.graph_objects as go
import plotly.colors
from scipy import stats

In [2]:
daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_smoother_tests.zarr')
MABN = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_smoother_tests.zarr')
GB = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_smoother_tests.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_smoother_tests.zarr')
GOME = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_smoother_tests.zarr')

In [3]:
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

In [4]:
summary_MABS = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\MABS_Summary_Stats.csv')
summary_MABN = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\MABN_Summary_Stats.csv')
summary_GB = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GB_Summary_Stats.csv')
summary_GOMW = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GOMW_Summary_Stats.csv')
summary_GOME = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GOME_Summary_Stats.csv')
bloom_MABS = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\MABS_Bloom_Metrics.csv')
bloom_MABN = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\MABN_Bloom_Metrics.csv')
bloom_GB = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GB_Bloom_Metrics.csv')
bloom_GOMW = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GOMW_Bloom_Metrics.csv')
bloom_GOME = pd.read_csv(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\GOME_Bloom_Metrics.csv')

## Part 1: Bloom Detection Method Testing

### Threshold Method

#### Function for determining the climatological threshold value

In [5]:
def threshold_value(thld=0.1, path=None):
    """
    Calculates the threshold value for chlorophyll-a based on a median baseline provided by the regional climatology.

    If no file path is provided, the path defaults to grabbing and reading the annual climatology file for the Northeast Shelf (NES) region. 
    The threshold is calculated by finding the percentage above the climatological CHL median for each pixel in the region.

    Args:
        thld (float, optional): The fraction value of the percentage above the median. This value defaults to 0.1 (10%).
        path (str, optional): The path to netCDF file used to calculate the threshold value. Defaults to None.

    Returns:
        xarray.DataArray: A spatial array containing the threshold values for each coordinate based on the median CHL value
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return thld_value

In [6]:
def spatial_threshold_value(shapefile_geometry=None,thld=0.1,path=None,regions=None,region_col='Region',ordered_region_names=None,default_shapefile_path='https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip'):
    """
    Creates a threshold value for a spatially averaged area.

    Using the threshold value, this function creates a spatially averaged threshold value for a region for use in other analysis.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, optional): Shapefile of the region in question. Defaults to None
        thld (float, optional): Percentage for threshold calcultion. Defaults to 0.1.
        path (str, optional): Path to climatology file. Defaults to None.
        regions (list or str, optional): A subset of region names to filter by. Defaults to None
        region_col (str, optional): The column name of the shapefile containing region names.
        ordered_region_names (list, optional): The list of names of the region in the shapefile if not already a column in the shapefile. Defaults to None
        default_shapefile_path (str, optional): Path to the default shapefile
    
    Returns:
        float. Value of the regionally averaged chlorophyll threshold.
    """
    # STEP 1: Load the shapefile geometry
    if shapefile_geometry is None:
        shapefile = gpd.read_file(default_shapefile_path)
        ordered_regions = ['Middle Atlantic Bight South','Middle Atlantic Bight North', 'Georges Bank', 'Gulf of Maine West', 'Gulf of Maine East']
        shapefile['Region'] = ordered_regions
    else:
        shapefile = shapefile_geometry.copy()
    if shapefile.crs is None:
        shapefile = shapefile.set_crs("EPSG:4326")
    else:
        shapefile = shapefile.to_crs("EPSG:4326")
    if ordered_region_names is not None:
        if len(ordered_region_names) != len(shapefile):
            raise ValueError("The list of names provided does not match the number of rows in the shapefile")
        shapefile['Region'] = ordered_region_names
        region_col = 'Region'

    # STEP 2: Subset the shapefile
    if regions is not None:
        if isinstance(regions,str):
            regions = [regions]
        shapefile = shapefile[shapefile[region_col].isin(regions)]
        if shapefile.empty:
            raise ValueError(f"None of the provided regions {regions} were found in the shapefile")
    
    # STEP 3: Load and prepare threshold data
    threshold = threshold_value(thld=thld,path=path)
    threshold.rio.write_crs("EPSG:4326",inplace=True)
    threshold.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    
    # STEP 4: Build the Dataset
    results = []
    for _, row in shapefile.iterrows():
        region_name = row[region_col]
        region_geometry = [mapping(row.geometry)]
        try:
            clipped_thld = threshold.rio.clip(region_geometry, shapefile.crs, drop=True)
            clipped_thld = clipped_thld.mean(dim=['lat','lon']).item()
            results.append({'Region':region_name, 'Threshold': clipped_thld})
        except Exception as e:
            print(f"Skipping {region_name} due to processing error")
            results.append({'Region': region_name, 'Threshold': None})

    return pd.DataFrame(results)

In [7]:
threshold_10 = spatial_threshold_value()
median_climatology = spatial_threshold_value(thld=0)

📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL


In [8]:
MABS_thld = threshold_10['Threshold'][0]
MABN_thld = threshold_10['Threshold'][1]
GB_thld = threshold_10['Threshold'][2]
GOMW_thld = threshold_10['Threshold'][3]
GOME_thld = threshold_10['Threshold'][4]
MABS_median = median_climatology['Threshold'][0]
MABN_median = median_climatology['Threshold'][1]
GB_median = median_climatology['Threshold'][2]
GOMW_median = median_climatology['Threshold'][3]
GOME_median = median_climatology['Threshold'][4]

#### Plotting the Climatological Threshold

In [ ]:
shapefile_geometry = [MAB_south_loc, MAB_north_loc, GB_whole_loc, GOM_west_loc, GOM_east_loc]
shapefile_names = ['MAB South', 'MAB North', 'Georges Bank', 'GOM West', 'GOM East']
colors = ['gold', 'cyan', 'darkorange', 'mediumorchid', 'dodgerblue']
clim_med = threshold_value(thld=0).squeeze()
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
fig = plt.figure(figsize=(20,8))
map_projection = cartopy.crs.PlateCarree()
ax = plt.axes(projection=map_projection)
im = plt.pcolormesh(clim_med.lon,
                    clim_med.lat,
                    clim_med,
                    cmap=cmocean.cm.algae,
                    norm=LogNorm(vmin=0.1, vmax=10.0)
)
custom_ticks = [0.1,1,10]
cb = plt.colorbar(im,shrink=0.8,ticks=custom_ticks,format='%g',pad=0.01) #$ $ makes it a LaTEX function so it actually formats as an equation
cb.set_label(label='Chlorophyll a Concentration ($mg/m^3$)',fontsize=12)
black_halo = [path_effects.Stroke(linewidth=4,foreground='black'), path_effects.Normal()]
white_halo = [path_effects.Stroke(linewidth=4,foreground='white'), path_effects.Normal()]
MAB_south_loc.boundary.plot(ax=ax, color='gold', linewidth=3, label="Middle Atlantic Bight South", path_effects=black_halo)
MAB_north_loc.boundary.plot(ax=ax, color='cyan', linewidth=3, label="Middle Atlantic Bight North", path_effects=black_halo)
GB_whole_loc.boundary.plot(ax=ax, color='darkorange', linewidth=3, label="Georges Bank", path_effects=black_halo)
GOM_west_loc.boundary.plot(ax=ax, color='mediumorchid', linewidth=3, label="Gulf of Maine West", path_effects=black_halo)
GOM_east_loc.boundary.plot(ax=ax, color='dodgerblue', linewidth=3, label="Gulf of Maine East", path_effects=black_halo)
ax.set_extent([-77,-62,37,47])
ax.legend(fontsize=16,loc='lower right')
for gdf,title,color in zip(shapefile_geometry,shapefile_names,colors):
    for idx, row in gdf.iterrows():
        rep_point = row.geometry.representative_point()
        if title == 'MAB South':
            rot_angle = 60
            x_offset, y_offset = (16,-3)
        elif title == "Georges Bank" or title == "GOM East":
            rot_angle=0
            x_offset, y_offset = (4,-2)
        else:
            rot_angle=0
            x_offset, y_offset = (0,0)
        ax.annotate(text=title,
                    xy=(rep_point.x, rep_point.y),
                    xytext=(x_offset,y_offset),
                    textcoords='offset points',
                    horizontalalignment='center',
                    fontsize=14,
                    color = 'black',
                    zorder=500,
                    fontweight='bold',
                    rotation=rot_angle,
                    bbox = dict(
                        boxstyle='round,pad=0.2',
                        facecolor='white',
                        alpha=0.5,
                        linewidth=1
                    )
        )
ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax.add_feature(cartopy.feature.LAND, zorder=50, facecolor='darkgrey')
ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree()) #Adding the shelf break line
states_provinces = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='50m',
    facecolor='none',
    edgecolor='gray',
    zorder = 100
)
ax.add_feature(states_provinces, linewidth=0.8)
ax.set_extent([-77,-65,35,45])
gl = ax.gridlines(crs=crs.PlateCarree(),draw_labels=True,color='dimgrey')
gl.top_labels = False
gl.right_labels = False
gl.xloactor = ticker.MultipleLocator(1)
gl.ylocator = ticker.MultipleLocator(1)
ax.set_title('Chlorophyll a Climatological Median', fontsize=20)
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\median_and_study_locations.png',dpi=300,bbox_inches='tight')

#### Create a mask to filter data for bloom conditions

In [9]:
def bloom_mask(path=None,thld=0.1,clim_path=None,Boolean=False): #Produces True and False values
    """
    Creates a mask on the chlorophyll-a data to only include values above the threshold set by the climatological median.

    If no file path is provided, the function searches for all D8 files (8 day rolling mean) for the Northeast Shelf. 
    If a file path is provided, it is currently set to open zarr files. The code exists to open netCDFs as well, it just needs to be uncommented. 
    If no climatology file path is provided, the function searches for the annual climatology file of the Northeast Shelf region.
    The median chlorophyll-a values are then extracted and compared to the threshold value found from the regional climatology.
    If the median chlorophyll-a values exceed the threshold, the value is stored as true. Otherwise, it is stored as false.

    Args:
        path (str, optional): Path to the daily data (or other temporal resolution data). Defaults to None
        thld (float, optional): Fractional value for percentage to calculate the climatological threshold per coordinate. Defaults to 0.1 (10%)
        clim_path (str, optional): Path to regional climatology file. Defaults to None
        Boolean (bool, optional): Determines how values are stored. False keeps the actual values and marks false values as 0. True stores an array of True and False values. Defaults to False

    Returns:
        xarray.DataArray: A spatial array with Boolean values or floats for each coordinate based on the threshold value.
    """
    if path is None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NES
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path)
    if Boolean is True:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    else:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        clim_med_new = clim_med.isel(time=0, drop=True) #Removes time dimension from climatological mean
        is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

#### Mask Intervals
Finding the climatological threshold at a few different inteverals (5%, 10%, 15%, 20%, 25%, and 30%)

In [ ]:
thld_value = [0.05,0.1,0.15,0.2,0.25,0.3]
clim_med = threshold_value()
bloom_5 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[1])
bloom_10 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[2])
bloom_15 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[3])
bloom_20 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[4])
bloom_25 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')
clim_med = threshold_value(thld_value[5])
bloom_30 = bloom_mask(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr')

Plotting all of the masks (5% - 30%)
<br> Must pick a day of the year from (INPUT VALUE RANGE HERE)

In [ ]:
DOY = 7181
datasets = [
    (bloom_5[DOY],"Chlorophyll a 5% Mask"),
    (bloom_10[DOY], "Chlorophyll a 10% Mask"),
    (bloom_15[DOY], "Chlorophyll a 15% Mask"),
    (bloom_20[DOY], "Chlorphyll a 20% Mask"),
    (bloom_25[DOY], "Chlorophyll a 25% Mask"),
    (bloom_30[DOY], "Chlorophyll a 30% Mask"),
    ]

fig, axes = plt.subplots(3,2,figsize=(14,12),subplot_kw={"projection":map_projection})
axes_flat = axes.flatten()

im= None

for i, (data,title) in enumerate(datasets):
    ax = axes_flat[i]
    im = ax.pcolormesh(data.lon,
                data.lat,
                data,
                cmap=cmocean.cm.algae,
                norm=LogNorm(vmin=0.1, vmax=10.0)
    )
    ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
    ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
    ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree())
    ax.set_xlabel('Longitude ($^o$)', fontsize=12)
    ax.set_ylabel('Latitude ($^o$)', fontsize=12)
    ax.set_extent([-77,-63,34.5,46])
    ax.set_title(title, fontsize=14) #Plot headings
    gl = ax.gridlines(draw_labels=True)
    gl.top_labels = False
    gl.right_labels = False

custom_ticks = [0.1,1,10]
cb = fig.colorbar(im,ax=axes,shrink=0.5,label='Chlorophyll a Concentration ($mg/m^3$)',ticks=custom_ticks,format='%g')
fig.suptitle("Chlorophyll a Masks Based on NES Annual Climatology",fontsize=20) #Overall figure heading

#### Histograms of Data

Create the functions to subset the data

In [10]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the data to a specified box for more precise spatial analysis.

    If no file path is provided, the function opens all D8 files for the Northeast Shelf. Currently it opens the D8_combined zarr file but can be changed to open the netCDF files.
    This function takes the daily data and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the daily data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median values for the spatial averaged area for the full time series of the data.
    """
    if path is None:
        #daily_data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        daily_data = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    daily_data_local = daily_data.CHL_median.sel(#Clips the CHL_median data to the specified spatial bounds
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    daily_data_local = daily_data_local.mean(dim=['lat','lon']) #Averages the data over the spatial bounds
    return daily_data_local

In [11]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the climatology data to a specified box for more precise spatial analysis.

    If no file path is provided, the function searches for the annual climatology file for the Northeast Shelf.
    This function takes the median chlorophyll-a of the climatology and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the climatology data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median value for the spatial averaged area for the climatology.
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL')
        clim = xr.open_dataset(file[0])
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded


In [12]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    """
    Creates a polygon shape for mapping

    This function takes in boundary coordinates and makes a shape to be used for plotting. 

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default

    Returns: 
        POLYGON
    """
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

#### Determining the percentage of datapoints that lie above each threshold

In [13]:
def percent_above_thld(data,clipped_thld):
    """
    Calculates the percentage of data points that lie above a threshold value.
    
    The data provided must be clipped to a region prior to inputting into function. Otherwise the function will run it for the entire spatial data in the dataset.

    Args: 
        data (xarray.Dataset, required): Dataset for analysis. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for region

    Returns:
        float. The percentage of datapoints that lie above the threshold.
    """
    dataset = data['CHL_median']
    threshold = clipped_thld
    total_above = int((dataset>threshold).sum())
    percent = (total_above/len(data))*100
    return percent

#### Shapefile Analysis

Clip the climatology data to the shapefile

In [106]:
clim_regional = xr.open_dataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL\ANNUAL_1998_2020-OCCCI-CHL-NES-STATS.nc')
clim_regional.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
clim_regional.rio.write_crs("epsg:4326", inplace=True)
clipped_MAB_s = clim_regional.rio.clip(MAB_south_loc.geometry, shapefile.crs, drop=True)
clim_MAB_s = clipped_MAB_s.CHL_median.mean(dim=['lat','lon'])
clipped_MAB_n = clim_regional.rio.clip(MAB_north_loc.geometry, shapefile.crs, drop=True)
clim_MAB_n = clipped_MAB_n.CHL_median.mean(dim=['lat','lon'])
clipped_GB = clim_regional.rio.clip(GB_whole_loc.geometry, shapefile.crs, drop=True)
clim_GB = clipped_GB.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_w = clim_regional.rio.clip(GOM_west_loc.geometry, shapefile.crs, drop=True)
clim_GOM_w = clipped_GOM_w.CHL_median.mean(dim=['lat','lon'])
clipped_GOM_e = clim_regional.rio.clip(GOM_east_loc.geometry, shapefile.crs, drop=True)
clim_GOM_e = clipped_GOM_e.CHL_median.mean(dim=['lat','lon'])
clim_NES = clim_regional.rio.clip(NES.geometry, shapefile.crs, drop=True)
clim_NES = clim_NES.CHL_median.mean(dim=['lat','lon'])

Clip D8 data to the shapefiles

In [107]:
daily_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
daily_data.rio.write_crs("epsg:4326", inplace=True)
clipped_daily_MABS = daily_data.rio.clip(MAB_south_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_south = clipped_daily_MABS.CHL_median.mean(dim=['lat','lon'])
clipped_daily_MABN = daily_data.rio.clip(MAB_north_loc.geometry.apply(mapping), shapefile.crs, drop=True)
MAB_north = clipped_daily_MABN.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GB = daily_data.rio.clip(GB_whole_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GB_whole = clipped_daily_GB.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOMW = daily_data.rio.clip(GOM_west_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_west = clipped_daily_GOMW.CHL_median.mean(dim=['lat','lon'])
clipped_daily_GOME = daily_data.rio.clip(GOM_east_loc.geometry.apply(mapping), shapefile.crs, drop=True)
GOM_east = clipped_daily_GOME.CHL_median.mean(dim=['lat','lon'])
clipped_daily_NES = daily_data.rio.clip(NES.geometry.apply(mapping), shapefile.crs, drop=True)
NES_full = clipped_daily_NES.CHL_median.mean(dim=['lat','lon'])

In [108]:
local_chl = [MAB_south,MAB_north,GB_whole,GOM_west,GOM_east,NES_full]
clim_med = [clim_MAB_s,clim_MAB_n,clim_GB,clim_GOM_w,clim_GOM_e,clim_NES]
clim_5 = [clim_MAB_s*1.05,clim_MAB_n*1.05,clim_GB*1.05,clim_GOM_w*1.05,clim_GOM_e*1.05,clim_NES*1.05]
clim_10 = [clim_MAB_s*1.10,clim_MAB_n*1.10,clim_GB*1.10,clim_GOM_w*1.10,clim_GOM_e*1.10,clim_NES*1.10]
clim_15 = [clim_MAB_s*1.15,clim_MAB_n*1.15,clim_GB*1.15,clim_GOM_w*1.15,clim_GOM_e*1.15,clim_NES*1.15]
clim_20 = [clim_MAB_s*1.2,clim_MAB_n*1.2,clim_GB*1.2,clim_GOM_w*1.2,clim_GOM_e*1.2,clim_NES*1.2]
clim_25 = [clim_MAB_s*1.25,clim_MAB_n*1.25,clim_GB*1.25,clim_GOM_w*1.25,clim_GOM_e*1.25,clim_NES*1.25]
clim_30 = [clim_MAB_s*1.3,clim_MAB_n*1.3,clim_GB*1.3,clim_GOM_w*1.3,clim_GOM_e*1.3,clim_NES*1.3]

In [109]:
data = local_chl[1]
numpy_array = data.compute().values
clean_data = numpy_array.ravel()
clean_data = clean_data[~np.isnan(clean_data)]

In [110]:
data = local_chl[2]
numpy_array = data.compute().values
clean_data_1 = numpy_array.ravel()
clean_data_1 = clean_data_1[~np.isnan(clean_data_1)]

In [ ]:
fig, ax = plt.subplots(nrows=1,ncols=1,figsize=(8,6))

ax.hist(clean_data_1,bins=100, range=(0.5,2.5))
ax.axvline(clim_med[2][0], color='red',label='Median',linewidth=3) #Pulls first value in list and then the 1 value in that value
ax.axvline(clim_5[2][0], color='mediumorchid', label='5% Threshold',linewidth=3)
ax.axvline(clim_10[2][0], color='gold',label='10% Threshold',linewidth=3)
ax.axvline(clim_15[2][0], color='darkorange',label='15% Threshold',linewidth=3)
ax.axvline(clim_20[2][0], color='midnightblue',label='20% Threshold',linewidth=3)
ax.axvline(clim_25[2][0], color='pink',label='25% Threshold',linewidth=3)
ax.axvline(clim_30[2][0], color='turquoise',label='30% Threshold',linewidth=3)
ax.set_title('Georges Bank',fontsize=18)
ax.set_ylabel("Number of Data Points",fontsize=12)
ax.set_xlabel("Chlorophyll a Concentration ($mg/m^3$)",fontsize=12)
fig.suptitle("Chlorophyll a Concentration Histograms",fontsize=20)
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Histograms_slideshow.png',dpi=300,bbox_inches='tight')

handles,labels = ax.get_legend_handles_labels()
fig_legend = plt.figure(figsize=(3,2))
ax_leg = fig_legend.add_subplot(111)

legend = ax_leg.legend(handles,labels,loc='center')
ax_leg.axis('off')
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Histograms_legend.png',dpi=300,bbox_inches='tight')

In [ ]:
fig, axes=plt.subplots(nrows=1,ncols=3,figsize=(16,5),sharey=True)

#Histograms
hist_axes = axes.flatten()
region_title = ["Middle Atlantic Bight North","Georges Bank",'Gulf of Maine West']
for i in range(len(region_title)):
    data = local_chl[i+1]
    ax = hist_axes[i]
    numpy_array = data.compute().values
    clean_data = numpy_array.ravel()
    clean_data = clean_data[~np.isnan(clean_data)]
    ax.hist(clean_data,bins=100, range=(0,4))
    ax.axvline(clim_med[i+1][0], color='red',label='Median') #Pulls first value in list and then the 1 value in that value
    ax.axvline(clim_5[i+1][0], color='mediumorchid', label='5% Threshold')
    ax.axvline(clim_10[i+1][0], color='gold',label='10% Threshold')
    ax.axvline(clim_15[i+1][0], color='darkorange',label='15% Threshold')
    ax.axvline(clim_20[i+1][0], color='midnightblue',label='20% Threshold')
    ax.axvline(clim_25[i+1][0], color='pink',label='25% Threshold')
    ax.axvline(clim_30[i+1][0], color='turquoise',label='30% Threshold')
    ax.set_xlabel("Chlorophyll a Concentration ($mg/m^3$)")
    ax.set_title(region_title[i])
handles, labels = hist_axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='lower center',
    ncol=3,
    fontsize=9,
    frameon=True,
    bbox_to_anchor=(0.5,-0.05),
)
plt.ylabel("Number of Data Points")
plt.tight_layout(rect=[0,0.07,1,0.95])
fig.suptitle("Chlorophyll a Concentration Histograms",fontsize=20,y=1.05)

#### Plotting the histograms centered on the median

Fix this function

In [14]:
def percent_deviation(dataset,clipped_median,clipped_thld):
    """
    Calculates the amount of data that is a certain percentage deviated from tmedian.

    Args:
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        clipped_median (float, required): The pre-calculated median for the region. No defaults
        clipped_thld (float, required): The pre-calculated threshold for the region. No defaults
    
    Returns:
        numpy.ndarray.
    """
    top = dataset.squeeze()-clipped_median
    fraction = top/clipped_thld
    percent_dev = fraction*100
    return percent_dev

In [93]:
MABS_per_dev = percent_deviation(MABS['CHL_median'].values,MABS_median,MABS_thld)
MABN_per_dev = percent_deviation(MABN['CHL_median'].values,MABN_median,MABN_thld)
GB_per_dev = percent_deviation(GB['CHL_median'].values,GB_median,GB_thld)
GOMW_per_dev = percent_deviation(GOMW['CHL_median'].values,GOMW_median,GOMW_thld)
GOME_per_dev = percent_deviation(GOME['CHL_median'].values,GOME_median,GOME_thld)

Plotting the histograms

In [ ]:
custom_bins=[0,5,10,15,20,25,30,35]
fig,axes=plt.subplots(2,3,figsize=(10,7),sharey=True)
axes=axes.flatten() #Creates a 1-D numpy array of indices for axes instead of a 2 by 3 array
data = [MABS_per_dev,MABN_per_dev,GB_per_dev,GOMW_per_dev,GOME_per_dev]
title = ['Middle Atlantic Bight South', 'Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
colors = ["gold",'cyan','darkorange','mediumorchid','dodgerblue']
for x in range(5):
    ax = axes[x]
    ax.hist(data[x], bins=custom_bins, facecolor = colors[x], edgecolor='black', linewidth=1)
    ax.set_xlabel("Percent Threshold")
    ax.tick_params(labelleft=True)
    ax.set_title(title[x])
axes[5].set_visible(False)
axes[0].set_ylabel("Amount of Data")
axes[3].set_ylabel("Amount of Data")
fig.suptitle("Percentage Threshold from Regional Median", fontsize=20)
plt.tight_layout()

### Rate of Change

#### Regional Rates of Change

In [15]:
def bounding_data(dataset,shapefile_geometry):
    """
    Regionally subsets a dataset for general analysis.

    This function takes a shapefile geometry and subsets a larger dataset to only include data within the shapefile. The data is averaged along the lat and lon dimensions.

    Args:
        dataset (xarray.Dataset, required): General dataset in question. No defaults
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults

    Returns:
        xarray.DataArray. The arrays of spatially sliced data.
    """
    dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    dataset.rio.write_crs("epsg:4326", inplace=True)
    clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile.crs, drop=True)
    regional_year = clipped_daily.CHL_median.mean(dim=['lat','lon'])
    return regional_year

In [16]:
def smoothing_data(shapefile_geometry=None,dataset = None, path=None,method="SavGol",window=15,poly=3,deriv=0,frac=0.00117):
    """
    Smoothes the raw chlorphyll-a data using a specific smoothing technique.

    If no method is provided, the default is the Savistky-Golay technique which has default parameters of a 15 day window and a polyorder of 3.
    If method is provided as "lowess", the frac value defaults to 0.00117, equivalent of a 12 day window on a 27 year time series.
    If no dataset path is provided, the function searches for D8 CHL files for the NES region. Currently, it opens the zarr file, but can be uncomment to open netCDFs.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        dataset (xarray.Dataset, optional): A spatially averaged dataset. Default is None
        path (str, optional): Path to a dataset. Defaults to daily D8 data for the full time series.
        method (str, optional): Smoothing technique applied. Defaults to "SavGol" but can also receive "lowess".
        window (int, optional): Window for SavGol smoothing. Default is 15
        poly (int, optional): polyorder for SavGol smoothing. Default is 3
        deriv (int, optional): Derivative of SavGol function. 0 provides smoothed data and 1 provides the first derivative. Defaults to 0 
        frac (float, optional): Frac value for lowess smoothing. Only necessary for using lowess smoothing. Default is 0.00117

    Returns:
        numpy.ndarray. Array of smoothed chlorophyll data values.
    """
    if dataset is not None:
        data = dataset
    elif path is None:
        #file = get_prod_files('CHL',map_region='NES',period='D8')
        #data = xr.open_mfdataset(file)
        data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_combined.zarr')
    else:
        data = xr.open_mfdataset(path)
    
    chl = data['CHL_median']
    time = data.time.astype('int64') #Changes time values to integers for smoothing 
    lowess_data = bounding_data(dataset,shapefile_geometry)
    lowess_median=lowess_data.to_dataframe() 
    lowess_median=lowess_median['CHL_median']
    if method == "SavGol":
        median = chl.values
        mask = ~np.isnan(median) #SavGol filter requires a dataset with no NaN values
        chl_interp = pd.Series(median).interpolate(method='linear').bfill().ffill().values #Linear interpolation to fill in NaN values
        sg_smoothed = savgol_filter(chl_interp,window_length=window,polyorder=poly,deriv=deriv) #Applying the SavGol filter
        sg_smoothed_full = np.copy(sg_smoothed)
        sg_smoothed_full[~mask]=np.nan #Puts the NaN values back in after smoothing
        smoothed_median = sg_smoothed_full[~mask]
    elif method == "lowess":
        if shapefile_geometry is None:
            print("Shapefile required for smoothing")
        smoothed_median = sm.nonparametric.smoothers_lowess.lowess(lowess_median,time,frac=frac) #Applying the lowess filter, lowess filter interpolates NaN values
    else:
        print("Error: Must specify smoothing technique")
    return smoothed_median

This function finds start and end dates of blooms

In [17]:
def bloom_peak_detection(clipped_thld,dataset=None,shapefile_geometry=None,window_for_peak=10,days=14,prm=0.1,**kwargs):
    """
    Detects all peak chlorophyll values that exceed the climatological threshold

    This function uses the smoothed chlorophyll data, identified peak values, and then masks that data to include only peaks that exceed the threshold set by the climatology.
    This function uses the find_peaks function from scipy, as well as the threshold_value() function and smoothing_data() function. 
    If no dataset is provided, the function smooths data based on the smoothing_data() function.
    The clipped_thld variable is created in the rolling_peak_window function or as a global variable.

    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold value for the region of interest. No defaults
        dataset (xarray.Dataset, optional): Already smoothed dataset. Defaults to None
        shapefile_geometry (geopandas.GeoDataFrame, optional): Shapefile of the region in question for smoothing. Defaults to None
        days (int, optional): Distance variable for scipy find_peaks. Distance allowed between consecutive peaks. Default is 14
        prm (float, optional): Prominence variable for find_peaks. Percent above the other peaks to be considered a peak. Default is 0.015
        **kwargs: Keywords for underlying functions. Expected keyword arguments include:
            - path (str, optional): Path to the data. Defaults to daily D8 data for the full time series.
            - method (str, optional): Smoothing technique. Defaults to "SavGol".
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only necessary if method == "lowess". Defaults to 0.00117
    
    Returns:
        List. List of days since the start of the dataset where the chlorophyll peaked and was above the threshold.
    """
    # STEP 1: Define the dataset. Uses a smoothed dataset (if provided). Else, it smooths the raw data provided for the region of interest.
    if dataset is None:
        smoothed_CHL = smoothing_data(shapefile_geometry,**kwargs)
    else:
        smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']

    # STEP 2: Find chlorophyll peaks with find peaks function.
        # Default of 14 days for distance was chosen after testing distances from 10-31. 10-20 separated peaks that never crossed below the threshold.
        # Default prominence of 0.1 captures major blooms while ignoring small peaks from daily fluctuations/sensor noise. Tested values in range of 0.01 - 0.2. 
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    chl_peaks = []
    chl_series = pd.Series(smoothed_CHL.values) #Turns chlorophyll values into a pandas series

    # STEP 3: Create a Boolean list for values that surpass/do not exceed the set threshold.
    is_above_threshold = chl_series>clipped_thld #Creates a true and false list. True if the value exceeds the threshold.

    # STEP 4: Searches Boolean list for places where the value switches from True to False (or False to True)
    change_from_prev_day = is_above_threshold != is_above_threshold.shift() #Checks if there is a change from previous day

    # STEP 5: Create streak IDs for each event and group events with the same ID together
    streak_IDs = change_from_prev_day.cumsum() #Creates ID for each event (New ID starts when the Boolean value changes. If no change, the ID is the same for that day)
    streak_lengths = is_above_threshold.groupby(streak_IDs).transform('sum') #Groups events together with the same ID and calculates the number of days that share that ID
    
    # STEP 6: Check to ensure the peak is above the threshold and check to see if its streak ID is >= to the defined window_for_peak.
        # If both conditions are true, we add it to the chl_peaks list. If one or both is not met, the peak is discarded.
    for peak in chl_peak_loc:
        peak_above_threshold = is_above_threshold[peak] #Checks that the peak is above the threshold
        peak_length = streak_lengths[peak]>=window_for_peak #Checks that the chlorophyll values remain above the threshold for a specified window
        if peak_above_threshold and peak_length:
            chl_peaks.append(peak)
    return chl_peaks

In [18]:
def bloom_event_detection(clipped_thld,chl_peaks_list,shapefile_geometry=None,dataset=None,event_distance=21,peak_window=10,verbose=False,**kwargs):
    """
    This function finds peaks in the chlorophyll-a time series and then groups together peaks in the same event based on proximity.

    If no dataset is provided, the function smooths out the data in the path given or the default data in the smoothing_data function. 
    If there are no peaks, the function returns an empty list.
    Using a list of chlorophyll peaks from the bloom_peak_detection function (previously calculated), the function searches for peaks in close proximity.
    A ten day rolling window is created for each peak to see if the chlorophyll value drop below the climatological threshold. If it does, the loop breaks.
    If peaks are too close together or the chlorophyll value does not drop below the threshold, they are considered one event. If these conditions are not met, they are separate events.

    Args:
        clipped_thld (variable, required): Threshold value for the region of interest. No defaults
        chl_peaks_list (list, required): Pre-calculated list of chlorophyll peaks for dataset. No defaults
        shapefile_geometry (geopandas.GeoDataFrame, optional): Shapefile of the region in question for smoothing. Defaults to None
        dataset (variable, optional): Already smoothed dataset. Default is None
        event_distance (int, optional): The number of days peaks must be apart to be considered separate events. Defaults to 21
        peak_window (int, optional): The number of days the chlorophyll concentration must remain above or below the threshold. Defaults to 10
        **kwargs: Additional arguments for bloom_peak_detection function. Inputs could include:
            - days (int, optional): Distance for find_peaks function. Defaults to 14
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.1
            - method (str, optional): Smoothing method for smoothing_data function. Defaults to "SavGol"
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): Polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only need if method == "lowess". Defaults to 0.00117
    Returns: 
        List: List of bloom event and peaks within each event.
    """
    # STEP 1: Finding peaks and the threshold value
    chl_peaks = chl_peaks_list
    if not chl_peaks: #Returns empty list if no peaks were found
        return []

    # STEP 2: Defines the smoothed dataset
    if dataset is None:
        smoothed_CHL = smoothing_data(shapefile_geometry,**kwargs)
    else:
        smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']
        smoothed_CHL = smoothed_CHL.values
    bloom_events = []

    # STEP 3: Creates the range for chlorophyll values to be observed in and identifies peak timeline
    current_event = [chl_peaks[0]] #Current event starts at the first peak identified
    days_between_events = event_distance #The number of days that must pass between conditions for the peaks to be considered separate events
    for i in range(1,len(chl_peaks)):
        previous_peak = chl_peaks[i-1] #Finds the previous peak
        current_peak = chl_peaks[i]
        chl_between_peaks = smoothed_CHL[previous_peak:current_peak] #Creates a list of all chlorophyll values between the current peak and previous peak
        dropped_below_thld = False

    # STEP 4: Find if the chlorophyll concentration drops below the threshold for a certain number of consecutive days
        #Checks to see if the number of days between chlorophyll peaks is above the specified peak window
        if len(chl_between_peaks)>=peak_window:
            chl_series = pd.Series(chl_between_peaks)
            #If all chlorophyll values are below the pre-determined threshold, dropped_below_thld is true. It adds up the trues and falses and finds the spots where the value is equal to peak_window
            dropped_below_thld = (chl_series<clipped_thld).rolling(window=peak_window).sum().eq(peak_window).any()
        if verbose is True:
            print(f"\n--- Checking transition from peak at index {previous_peak} to {current_peak} ---")
            print(f"Distance between peaks: {current_peak - previous_peak} (Needs to be >= {days_between_events} to split based on time)")
            print(f"Number of data points in gap: {len(chl_between_peaks)}")
            print(f"Minimum CHL value in gap: {min(chl_between_peaks):.3f} (Threshold is {clipped_thld:.3f})")
            print(f"Did it stay below threshold for {peak_window} consecutive points? {dropped_below_thld}")
    # STEP 5: Append events to events list. 
        if current_peak-previous_peak<days_between_events or not dropped_below_thld: #If peaks are too close together or does not drops below threshold, they are the same event.
            current_event.append(current_peak)
        else: #Peaks are an appropriate distance apart or chl drop below the threshold.
            bloom_events.append(current_event)
            current_event = [current_peak]
    bloom_events.append(current_event)
    return bloom_events


In [19]:
def max_peak(event, dataset):
    """
    Finds the peak in an event with the maximum chlorophyll concentration for the event.
    For use in bloom_timing function.

    Args:
        event (list, required): Pre-calculated list of peaks for the event. No defaults
        dataset (xarray.Dataset, required): The dataset for analysis. No defaults

    Returns:
        List. A list of peaks associated with that blooms maximum chlorophyll concentration.
    """
    region_smoothed = dataset['smoothed_sg_win_15_poly_3']
    region_smoothed_values = region_smoothed.values #Extracts chl-a values
    region_time_smoothed = dataset['time'].values #Extracts time values
    peak_chl_values = -float('inf') 
    max_chl_day = None
    flatten_event = [] #For events that are multimodal, this flattens it into one list and not a tuple
    for item in event:
        if isinstance(item,(tuple,list,np.ndarray)): #If the event has more than one peak, it makes it one list and not a variety of data types.
            flatten_event.extend(item)
        else:
            flatten_event.append(item)
    for day in flatten_event:
        chl_peak = region_smoothed_values[int(day)] #Finds the chl value at the peak
        if chl_peak>peak_chl_values: #Checks to see if the current chl value is greater than the previous peak's value.
            peak_chl_values = chl_peak #If it is, it becomes the new maximum of the event
            max_chl_day = day #This is the day of the maximum
    peak_DOY = region_time_smoothed[max_chl_day]
    peak_chl = region_smoothed.values[max_chl_day]
    ts = pd.Timestamp(peak_DOY)
    peak_date = ts.date()
    peak_doy = ts.dayofyear
    return peak_date,peak_doy,peak_chl

In [20]:
def max_roc_for_bloom(start_DOY,end_DOY,dataset):
    """
    Finds the maximum rate of change for each bloom.

    This function uses a pre-saved dataset of daily rates of change and pre-calculated start and end DOYs for each event
    It then finds the maximum rate of change between the initiation and termination date and then adds it to the start day value to get the DOY value for the maximum rate of change.
    This function is part of the bloom_timing function.

    Args:
        start_DOY (int, required): Pre-calculated start DOY for the event. No defaults
        end_DOY (int, required): Pre-calculated end DOY for the event. No defaults
        dataset (xarray.Dataset, required): Dataset of interest. No defaults
    Returns:
        int/float: The maximum rates of change for the bloom.
    """
    roc = dataset['ROC_SG']
    range_roc = roc[start_DOY:end_DOY+1]
    if len(range_roc)>0:
        range_max_roc = np.nanargmax(range_roc) #Finds local maximum rate of change for each detected bloom
        max_roc = start_DOY+range_max_roc #Gets the actual day of year value
    max_roc = max_roc
    return max_roc

In [21]:
def bloom_timing(clipped_thld,clipped_med,dataset,init_term_window=5,trough_length=3,search_window=90,**kwargs):
    """
    This function finds the initiation and termination dates of blooms based on a rolling peak window.

    This function finds a variety of bloom timing values, including:
        - Bloom events: The events (and peaks within the events) throughout the dataset that satisfy the peak conditions.
        - Start day: The day since the start of the dataset where the initiation conditions were met for each bloom event.
        - End day: The day since the start of the dataset where the termination conditions were met for each bloom event
        - Peak date: The date of the maximum chlorophyll peak for each bloom event.
        - Peak DOY: The DOY (1-366) of the maximum chlorophyll peak for each bloom event.
        - Maximum chlorophyll: The maximum chlorophyll value for each bloom event.
        - Maximum rate of change: The maximum rate of change for each bloom event.
        - First exceedance day: The day for each bloom event where the chlorophyll concentration first exceed the pre-determined threshold.
        - Last dip day: The day for each bloom event where the chlorophyll concentration last dipped below the pre-determined threshold.


    Args:
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults
        dataset (xarray.Dataset, required): Already smoothed dataset. No defaults
        init_term_window (int, optional): Amount of time each condition must be met for it to trigger an initiation or termination date. Defaults to 5
        trough_length (int, optional): The number of days on either side of a minimum chl value that the value must remain below for it to be considered a trough. Defaults to 3
        search_window (int, optional): The number of days after the final peak of an event that the function searches through to find the termination date. Defaults to 90
        **kwargs: Additional input for bloom_event_detection and threshold_value functions. Possible inputs include: 
            - peak_window (int, optional): The amount of time a peak must remain above the threshold for it to be considered an event. Defaults to 10  
            - days (int, optional): Distance for find_peaks function. Defaults to 10
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.015
            - method (str, optional): Smoothing method for smoothing_data function. Defaults to "SavGol"
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): Polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only need if method == "lowess". Defaults to 0.00117
    Returns:
        tuple: A tuple containing (start_DOY, end_DOY, merge_bloom_events, peak_dates, peak_DOYs, maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list), where:
            start_DOY (list): The list of bloom initiation days.
            end_DOY (list): The list of bloom termination days.
            merge_bloom_events (list): The list of bloom events.
            peak_dates (list): The list of chlorophyll maximum days for each bloom event.
            peak_DOYs (list): The list of dates of the chlorophyll maximums for each bloom event.
            maximum_chl (list): The list of maximum chlorophyll values for each bloom event.
            maximum_roc (list): The list of maximum rates of change for each bloom event.
            start_at_thld_list (list): The list of DOYs where the chl first crosses the threshold for an event.
            end_at_thld_list (list): The list of DOYs where the chl last dipped below the threshold for an event.
    """
    # STEP 1: Find the climatological median, rate of change, and bloom events
    chl_median = dataset['smoothed_sg_win_15_poly_3'].to_series().reset_index(drop=True)
    roc = dataset['ROC_SG'].to_series().reset_index(drop=True)
    chl_peaks_list = bloom_peak_detection(clipped_thld=clipped_thld,dataset=dataset)
    bloom_events = bloom_event_detection(dataset=dataset,clipped_thld=clipped_thld,chl_peaks_list=chl_peaks_list,**kwargs)
    
    # STEP 2: Define peak windows
    last_end_day, last_start_day = 0, 0
    start_DOY, end_DOY, merge_bloom_events = [], [], []
    peak_dates, peak_DOYs, maximum_chl = [], [], []
    maximum_roc = []
    max_index = len(roc)-1

    # STEP 3: Identify all possible initiation and termination dates for the full time series
    is_roc_negative =  roc < 0
    is_roc_positive = roc >= 0
    is_below_threshold = chl_median < clipped_thld
    is_below_median = chl_median <= clipped_med

    #Find all days for time series where initiation conditions are met (positive growth and below the threshold)
    initiation_conditions_met = is_roc_positive & is_below_threshold
    initiation_rolling = initiation_conditions_met.rolling(window=init_term_window).sum()
    initiation_days = initiation_rolling[initiation_rolling == init_term_window].index.to_numpy()

    #Find all days for time series where termination conditions are met
    termination_conditions_met = is_roc_negative & is_below_threshold
    termination_rolling = termination_conditions_met.rolling(window=init_term_window).sum()
    termination_days = termination_rolling[termination_rolling == init_term_window].index.to_numpy()

    #Find all local troughs for the full dataset. The troughs must have chl values lower than specified consecutive days on either side. 
    #This smooths out some of the smaller bumps caused by the noisy chl-a data and keeps major troughs. Tested 1,2,3. 
    local_minimum = (chl_median < chl_median.shift(trough_length)) & (chl_median < chl_median.shift(-trough_length)) & is_below_median
    local_minimum = local_minimum[local_minimum].index.to_numpy()

    # STEP 4: Find the initiation date of the bloom based on the rate of change.
    for event in bloom_events:
        event_start = event[0]
        event_end = event[-1]

        #Sets the end of the window to be 180 days from the last peak in the event or the end of the dataset, whichever comes first.
        end_of_window = min(max_index,event[-1]+search_window) 
        start_day = last_end_day #Sets the start of the window to the last end day

        #Find all possible initiation days between the end of the last bloom and the first peak in the current event.
        possible_init_dates = initiation_days[(initiation_days < event_start)&(initiation_days >= start_day)] 

        if len(possible_init_dates) > 0 and last_end_day >= last_start_day: #Ensures that the initiation date is not before the previous bloom's termination date
            possible_start_day = possible_init_dates[-1] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= last_end_day) & (local_minimum <= event_start)] #Finds all troughs between first peak and the previous termination

            #Attach it to the closest trough if one is available
            if len(window_troughs) > 0:
                 closest_trough = np.abs(window_troughs - possible_start_day).argmin() #Finds the closest trough to the first peak
                 start_day = window_troughs[closest_trough] 
            else:
                 start_day = possible_start_day  

    # STEP 5: Find the termination date 
        end_day = event_end
        possible_term_dates = termination_days[(termination_days>event_end) & (termination_days<=end_of_window)]

        if len(possible_term_dates)>0:
            possible_end_day = possible_term_dates[0] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
            if len(window_troughs) > 0:
                closest_trough = np.abs(window_troughs - possible_end_day).argmin() #Finds the closest trough to the first peak
                end_day = window_troughs[closest_trough]

        else: #If termination conditions are not met for a bloom, find the next trough that is below the threshold value and make that the termination date.
                potential_trough = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
                if len(potential_trough) > 0:
                    end_day = potential_trough[0]

    # STEP 6: Find peak DOY, date, and chlorophyll values for each bloom event.
        peak_date, peak_DOY, max_chl = max_peak(event,dataset)

    # STEP 7: Find the maximum rate of change for the bloom event
        max_roc = max_roc_for_bloom(start_DOY=start_day,end_DOY=end_day,dataset=dataset)

    # STEP 8: Merge and/or append events to the lists
        same_bloom = len(start_DOY) > 0 and start_day == start_DOY[-1] and end_day == end_DOY[-1]
        same_timing = (last_end_day>0) and (event[0]<=last_end_day)
        if same_bloom or same_timing: #This ensures that termination dates are not duplicated and every initiation date has a termination date
            merge_bloom_events[-1].extend(event)
            if end_day>end_DOY[-1]:
                end_DOY[-1] = end_day
                if max_chl > maximum_chl[-1]:
                    maximum_chl[-1] = max_chl
                    peak_dates[-1] = peak_date
                    peak_DOYs[-1] = peak_DOY
                #Recalculate the maximum ROC for the expanded timeline
                new_max_roc = max_roc_for_bloom(start_DOY=start_DOY[-1],end_DOY=end_DOY[-1],dataset=dataset)
                maximum_roc[-1] = new_max_roc

        else:
            #Documented as an entirely new event
            start_DOY.append(start_day)
            end_DOY.append(end_day)
            merge_bloom_events.append(list(event))
            peak_dates.append(peak_date)
            peak_DOYs.append(peak_DOY)
            maximum_chl.append(max_chl)
            maximum_roc.append(max_roc)
        
        last_start_day = start_day #Resets start and end dates for the loop
        last_end_day = end_day

    # STEP 9: Find the days where the chl first exceeds the threshold and where it last dips below the threshold
    chl_median_values = dataset['smoothed_sg_win_15_poly_3'].values
    start_at_thld_list = []
    end_at_thld_list = []
    for i in range(len(start_DOY)):
        start = start_DOY[i]
        peak_start = merge_bloom_events[i][0]
        #Look forward for the day where it first crosses above the threshold
        start_found = False
        for j in range(start + 1, peak_start + 1, 1):
            if chl_median_values[j] >= clipped_thld:
                start_at_thld_list.append(j)
                start_found = True
                break
        if not start_found:
            start_at_thld_list.append(peak_start)

        end = end_DOY[i]
        peak_end = merge_bloom_events[i][-1]
        #Look backwards for the last drop below the threshold
        end_found = False
        if end > peak_end:
            for j in range (end, peak_end - 1, -1):
                if chl_median_values[j] >= clipped_thld:
                    end_at_thld_list.append(j+1)
                    end_found = True
                    break
        if not end_found:
            end_at_thld_list.append(end)
    return start_DOY,end_DOY,merge_bloom_events,peak_dates,peak_DOYs,maximum_chl, maximum_roc,start_at_thld_list,end_at_thld_list

In [ ]:
#time_series = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
time_series =[2007]
title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
file = ["MABS","MABN","GB","GOMW","GOME"]
data_regions = [MABS,MABN,GB,GOMW,GOME]
shapefile_location = [MAB_south_loc,MAB_north_loc,GB_whole_loc,GOM_west_loc,GOM_east_loc]
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
#plt.ioff()
import time
for x in range(5):
    print(f"--- Starting Region {title[x]} ---")
    region_title = title[x]
    raw_data=data_regions[x]
    t0 = time.time()
    start_DOY,end_DOY,bloom_events,_,_,_,max_roc,_,_ = bloom_timing(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_regions[x])
    #print(f"max_roc_for_bloom took: {time.time() - t0:.2f} seconds")
    t1 = time.time()
    #print(f"spatial_threshold_value took: {time.time() - t1:.2f} seconds")
    smoothed = raw_data
    raw_time = pd.to_datetime(raw_data['time'].values)
    time_smoothed = smoothed['time']
    time_smoothed = pd.to_datetime(time_smoothed)
    chl_smoothed = smoothed['smoothed_sg_win_15_poly_3']
    roc = smoothed['ROC_SG']
    raw_chl = raw_data['CHL_median'].values

    #Converting ROC day of year into date format
    roc_max_time = np.asarray(max_roc).astype(int)
    roc_max_date = pd.to_datetime(time_smoothed[roc_max_time])
    roc_max = chl_smoothed[roc_max_time]

    #Bloom start and end date
    start_DOY_time = np.asarray(start_DOY).flatten().astype(int)
    start_DOY_date = pd.to_datetime(time_smoothed[start_DOY_time])
    start_DOY_val = chl_smoothed[start_DOY_time]
    end_DOY_time = np.asarray(end_DOY).flatten().astype(int)
    end_DOY_date = pd.to_datetime(time_smoothed[end_DOY_time])
    end_DOY_val = chl_smoothed[end_DOY_time]
    bloom_peak_time = np.concatenate(bloom_events if len(bloom_events)>0 else [np.asarray([])]).astype(int)
    bloom_peak_date = pd.to_datetime(time_smoothed[bloom_peak_time])
    bloom_peak_val = chl_smoothed[bloom_peak_time]
    
    for year in time_series:
        #print(f"  Plotting {year}...")
        x_left = year-1
        x_right = year+1
        start_bound = pd.to_datetime(f'{x_left}-01-01')
        end_bound = pd.to_datetime(f'{x_right}-01-01')

        mask_raw = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
        mask_smooth = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
        mask_roc = (roc_max_date >= start_bound) & (roc_max_date < end_bound)
        mask_start = (start_DOY_date >= start_bound) & (start_DOY_date < end_bound)
        mask_end = (end_DOY_date >= start_bound) & (end_DOY_date < end_bound)
        mask_peak = (bloom_peak_date >= start_bound) & (bloom_peak_date < end_bound)

        #Plotting
        #t2 = time.time()
        fig=plt.figure(figsize=(18,10))
        plt.plot(raw_time[mask_raw].values,raw_chl[mask_raw],label="Raw Chl-a") #Plots the time series of raw data
        plt.plot(time_smoothed[mask_smooth],chl_smoothed[mask_smooth], label="Smoothed Chl-a")
        plt.axhline(clipped_median[x],c="purple",label="Climatological median")
        plt.axhline(clipped[x],c="red",label="10% Threshold")
        plt.scatter(roc_max_date[mask_roc],roc_max[mask_roc],c='green',s=100,zorder=5,marker='^',label="Max rates of change")
        plt.ylabel("Chlorophyll a Concentrations ($mg/m^3$)")
        plt.xlabel("Date")
        plt.xlim(start_bound, end_bound)
        plt.title("Chlorophyll a in the " + region_title + " in " + str(year))
        plt.scatter(start_DOY_date[mask_start],start_DOY_val[mask_start],c='magenta',s=50,zorder=5,marker='s',label="Bloom start")
        plt.scatter(end_DOY_date[mask_end],end_DOY_val[mask_end],c='darkblue',s=75,zorder=5,marker='*',label="Bloom end")
        plt.scatter(bloom_peak_date[mask_peak],bloom_peak_val[mask_peak],c='crimson',s=50,zorder=5,marker='D',label="Bloom peak")
        plt.legend(fontsize=8)

        #axes_flat[1].plot(pd.to_datetime(time_smoothed),roc)
        #roc_max_values = roc[roc_max_time]
        #y=0
        #axes_flat[1].scatter(roc_max_date[mask_roc],roc_max_values[mask_roc],c='green',s=100,zorder=5,marker='^',label="Max rates of change")
        #axes_flat[1].axhline(y,c='purple')
        #axes_flat[1].set_xlim(start_bound, end_bound)
        #axes_flat[1].set_title("Rate of Change for " + region_title + " in " + str(year))
        #axes_flat[1].legend()
        #axes_flat[1].set_ylabel("Rate of Change per day")
        #axes_flat[1].set_xlabel("Date")

        filename = f"{str(file[x])}_{str(year)}_5.png"
        plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\5day_med_graphs\{filename}')
        plt.close(fig)
        #print(f"  Saved {year} image in: {time.time() - t2:.2f} seconds")
    print(f"Successfully finished {file[x]} region graphs")

In [40]:
time_series =[2007]
title = ["Middle Atlantic Bight South","Middle Atlantic Bight North","Georges Bank","Gulf of Maine West","Gulf of Maine East"]
file = ["MABS","MABN","GB","GOMW","GOME"]
data_regions = [MABS,MABN,GB,GOMW,GOME]
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
import time
x=2
region_title = title[x]
raw_data=data_regions[x]
t0 = time.time()
start_DOY,end_DOY,bloom_events,_,_,_,max_roc,_,_ = bloom_timing(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_regions[x])
#print(f"max_roc_for_bloom took: {time.time() - t0:.2f} seconds")
t1 = time.time()
#print(f"spatial_threshold_value took: {time.time() - t1:.2f} seconds")
smoothed = raw_data
raw_time = pd.to_datetime(raw_data['time'].values)
time_smoothed = smoothed['time']
time_smoothed = pd.to_datetime(time_smoothed)
chl_smoothed = smoothed['smoothed_sg_win_15_poly_3']
roc = smoothed['ROC_SG']
raw_chl = raw_data['CHL_median'].values

#Converting ROC day of year into date format
roc_max_time = np.asarray(max_roc).astype(int)
roc_max_date = pd.to_datetime(time_smoothed[roc_max_time])
roc_max = chl_smoothed[roc_max_time]

#Bloom start and end date
start_DOY_time = np.asarray(start_DOY).flatten().astype(int)
start_DOY_date = pd.to_datetime(time_smoothed[start_DOY_time])
start_DOY_val = chl_smoothed[start_DOY_time]
end_DOY_time = np.asarray(end_DOY).flatten().astype(int)
end_DOY_date = pd.to_datetime(time_smoothed[end_DOY_time])
end_DOY_val = chl_smoothed[end_DOY_time]
bloom_peak_time = np.concatenate(bloom_events if len(bloom_events)>0 else [np.asarray([])]).astype(int)
bloom_peak_date = pd.to_datetime(time_smoothed[bloom_peak_time])
bloom_peak_val = chl_smoothed[bloom_peak_time]

for year in time_series:
    #print(f"  Plotting {year}...")
    start_bound = pd.to_datetime(f'{year}-01-01')
    end_bound = pd.to_datetime(f'{year}-12-31')

    mask_raw = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
    mask_smooth = (time_smoothed >= start_bound) & (time_smoothed < end_bound)
    mask_roc = (roc_max_date >= start_bound) & (roc_max_date < end_bound)
    mask_start = (start_DOY_date >= start_bound) & (start_DOY_date < end_bound)
    mask_end = (end_DOY_date >= start_bound) & (end_DOY_date < end_bound)
    mask_peak = (bloom_peak_date >= start_bound) & (bloom_peak_date < end_bound)


    x_ticks = ['January','February','March','April','May','June','July','August','September','October','November','December']
    #Plotting
    #t2 = time.time()
    fig, axes=plt.subplots(nrows=2,ncols=1,figsize=(18,10),sharex=True)
    axes_flat = axes.flatten()
    axes_flat[0].plot(raw_time[mask_raw].values,raw_chl[mask_raw], linewidth=3, label="Raw Chl-a") #Plots the time series of raw data
    axes_flat[0].plot(time_smoothed[mask_smooth],chl_smoothed[mask_smooth], linewidth=3, label="Smoothed Chl-a")
    axes_flat[0].axhline(clipped_median[x],c="purple",label="Climatological median",linewidth=3)
    axes_flat[0].axhline(clipped[x],c="darkturquoise",label="10% Threshold",linewidth=3)
    axes_flat[0].scatter(roc_max_date[mask_roc],roc_max[mask_roc],c='green',s=200,zorder=5,marker='^',label="Max rates of change")
    axes_flat[0].set_ylabel("Chlorophyll a Concentrations ($mg/m^3$)",fontsize=15)
    axes_flat[0].set_xlim(start_bound, end_bound)
    axes_flat[0].xaxis.set_major_locator(mdates.MonthLocator())
    axes_flat[0].xaxis.set_major_formatter(mdates.DateFormatter('%B'))
    axes_flat[0].tick_params(labelbottom=True)
    axes_flat[0].set_title("Chlorophyll a in the " + region_title + " in " + str(year),fontsize=20)
    axes_flat[0].scatter(start_DOY_date[mask_start],start_DOY_val[mask_start],c='magenta',s=200,zorder=5,marker='s',label="Bloom start")
    axes_flat[0].scatter(end_DOY_date[mask_end],end_DOY_val[mask_end],c='mediumblue',s=225,zorder=5,marker='p',label="Bloom end")
    axes_flat[0].scatter(bloom_peak_date[mask_peak],bloom_peak_val[mask_peak],c='crimson',s=200,zorder=5,marker='D',label="Bloom peak")
    axes_flat[0].tick_params(axis='both',labelsize=12)

    axes_flat[1].plot(pd.to_datetime(time_smoothed),roc,linewidth=3)
    roc_max_values = roc[roc_max_time]
    y=0
    axes_flat[1].scatter(roc_max_date[mask_roc],roc_max_values[mask_roc],c='green',s=200,zorder=5,marker='^')
    axes_flat[1].axhline(y,c='purple',linewidth=3)
    axes_flat[1].set_xlim(start_bound, end_bound)
    axes_flat[1].set_title("Rate of Change for " + region_title + " in " + str(year),fontsize=20)
    axes_flat[1].set_ylabel("Rate of Change per day",fontsize=15)
    axes_flat[1].set_xlabel("Month",fontsize=15)
    axes_flat[1].set_ylim(-0.1,0.1)
    axes_flat[1].tick_params(axis='both',labelsize=12)

    filename = f"{str(file[x])}_{str(year)}_5.png"
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\{filename}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    #print(f"  Saved {year} image in: {time.time() - t2:.2f} seconds")
    handles,labels = axes_flat[0].get_legend_handles_labels()
    fig_legend = plt.figure(figsize=(4,3))
    ax_leg = fig_legend.add_subplot(111)

    legend = ax_leg.legend(handles,labels,loc='center',labelspacing=1)
    ax_leg.axis('off')
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\time_series_legend',dpi=300,bbox_inches='tight')
    plt.close(fig_legend)
print(f"Successfully finished {file[x]} region graphs")

Successfully finished GB region graphs


## Part 2: Quantify the Number of Phytoplankton Bloom Days Per Year

In [22]:
def bloom_classification(dataset,bloom_events):
    """
    Classifies identified blooms by the peak DOY as spring, fall, or other.

    This function classifies a bloom as a spring bloom, fall bloom, or other bloom based on its peak DOY. Based on the seasons and relative start and peak times, the DOY ranges are
        - 1 to 59 for winter blooms (January 1 to February 28)
        - 61 to 152 for spring blooms (March 1 to June 1)
        - 244 to 366 for fall blooms (September 1 to December 31)
    Other blooms do not fall within these DOY ranges. 

    Args:
        dataset (xarray.Dataset, required): Dataset of region of interest. No defaults
        bloom_events (list, required): A pre-calculated list of all bloom events for the dataset

    Returns:
        Tuple: A tuple containing all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms, where:
            all_blooms: A list of all blooms as their string classification "Spring", "Fall", or "Other".
            spring_blooms: A list of all spring blooms for the dataset. Returns the peak DOY. 
            fall_blooms: A list of all fall blooms for the dataset. Returns the peak DOY.
            winter_blooms: A list of all winter blooms for the dataset. Returns the peak DOY
            other_blooms: A list of blooms not classified as spring or fall blooms. Returns peak DOY.
    """
    region_time_smoothed = dataset['time'].values
    spring_blooms = []
    fall_blooms = []
    winter_blooms = []
    other_blooms = []
    all_blooms = []
    for event in bloom_events:
        initial_peak = event[0]
        initial_peak_date = region_time_smoothed[initial_peak]
        initial_peak_DOY = pd.to_datetime(initial_peak_date).dayofyear
        if initial_peak_DOY >= 60 and initial_peak_DOY <=152:
            potential_spring_bloom = initial_peak_DOY
            spring_blooms.append(potential_spring_bloom)
            all_blooms.append("Spring")
        #Identify fall blooms as last bloom of the year or within the fall DOY range
        elif initial_peak_DOY >= 245 and initial_peak_DOY <=366:
            potential_fall_bloom = initial_peak_DOY
            fall_blooms.append(potential_fall_bloom)
            all_blooms.append("Fall")
        #Identify other blooms as other
        elif initial_peak_DOY >=1 and initial_peak_DOY <=59:
            potential_winter_bloom = initial_peak_DOY
            winter_blooms.append(potential_winter_bloom)
            all_blooms.append("Winter")
        else:
            potential_other_bloom = initial_peak_DOY
            other_blooms.append(potential_other_bloom)
            all_blooms.append("Summer")
    return all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms

#### Actual number of bloom days per year over the time series

In [23]:
def bloom_days_per_year(dataset, year, start_DOY, end_DOY, thld_start=None, thld_end=None, type='calendar'):
    """
    Calculates the number of bloom days per year from bloom initiation to termination and that are above the threshold.

    This function uses the start_DOY and end_DOY lists from the bloom_timing function.
    Then it slices the data and calculates all of the bloom days for the specified year.
    This function assumes a start date of January 1 if the start date falls in the previous year. 
    It assumes a termination date of December 31 if the termination date is in the following year.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function. Defaults to end_DOY
        thld_start (list, required): The list of DOY where the chlorophyll first exceed the threshold.
        thld_end (list, required): The list of DOY where the chlorophyll last dipped below the threshold.
        type (str, optional): The year type for calculation. Accepts 'calendar' (Jan - Dec) and 'biological' (July - June)

    Returns:
        Int. The number of bloom days for the given year.
    """
    if type == "calendar":
        start_time = pd.to_datetime(f"{year}-01-01")
        end_time = pd.to_datetime(f"{year}-12-31")
    elif type == "biological":
        start_time = pd.to_datetime(f"{year}-07-01")
        end_time = pd.to_datetime(f"{year+1}-06-30")
    else:
        raise ValueError("Invalid year type. Choose 'calendar' or 'biological'.")
    # STEP 1: Total number of bloom days from initiation to termination
    number_bloom_days = []
    for x in range(len(start_DOY)):
        bloom_start_date = int(start_DOY[x])
        start_date = pd.to_datetime(dataset['time'].values[bloom_start_date])
        if x < len(end_DOY):
            bloom_end_date = int(end_DOY[x])
            end_date = pd.to_datetime(dataset['time'].values[bloom_end_date])
        else:
            end_date = end_time
        overlap_start = max(start_date, start_time)
        overlap_end = min(end_date, end_time)

        if overlap_start <= overlap_end:
            amount_bloom_days = (overlap_end - overlap_start).days+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_year=sum(number_bloom_days)

    # STEP 2: The number of days per year above the threshold during a bloom
    number_days_above_thld = []
    for x in range(len(thld_start)):
        thld_bloom_start_date = int(thld_start[x])
        thld_start_date = pd.to_datetime(dataset['time'].values[thld_bloom_start_date])
        if x < len(thld_end):
            thld_bloom_end_date = int(thld_end[x])
            thld_end_date = pd.to_datetime(dataset['time'].values[thld_bloom_end_date])
        else:
            thld_end_date = end_time
        thld_overlap_start = max(thld_start_date, start_time)
        thld_overlap_end = min(thld_end_date, end_time)

        if thld_overlap_start <= thld_overlap_end:
            thld_amount_bloom_days = (thld_overlap_end - thld_overlap_start).days+1
            number_days_above_thld.append(thld_amount_bloom_days)

    annual_days_above_thld = sum(number_days_above_thld)
    return bloom_days_per_year, annual_days_above_thld

In [24]:
def bloom_days_per_month(dataset, year, month, start_DOY, end_DOY):
    """
    Calculates the number of bloom days per month from bloom initiation to termination.

    This function uses the rolling_peak_window() function. It finds the initiation and termination for each date in the time series.
    Then it slices the data and calculates all of the bloom days for the specified month.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        month (int, required): The number of the month of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function. Defaults to end_DOy

    Returns:
        Int. The number of bloom days for the given year.
    """
    number_bloom_days = []
    _, last_day_month = calendar.monthrange(year,month)
    month_start = pd.Timestamp(year=year,month=month,day=1)
    month_end = pd.Timestamp(year=year, month=month,day=last_day_month)

    for x in range(len(start_DOY)):
        start_index = int(start_DOY[x])
        bloom_start_date = pd.to_datetime(dataset['time'].values[start_index])
        if x < len(end_DOY):
            end_index = int(end_DOY[x])
            bloom_end_date = pd.to_datetime(dataset['time'].values[end_index])
        else:
            bloom_end_date = pd.Timestamp(year=year,month=12,day=31)
        start_overlap = max(bloom_start_date,month_start)
        end_overlap = min(bloom_end_date,month_end)

        if start_overlap <= end_overlap:
            amount_bloom_days = (end_overlap-start_overlap).days+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_month=sum(number_bloom_days)
    return bloom_days_per_month

In [25]:
def percent_bloom_days(dataset, year, start_DOY, end_DOY, thld_start=None, thld_end=None):
    annual_bloom_days, annual_days_above_thld = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
    bloom_days_total = annual_bloom_days
    days_above_thld = annual_days_above_thld
    percent = days_above_thld/bloom_days_total
    return percent

#### Average number of bloom days over the time series

## Part 3: Quantify the Number of Phytoplankton Blooms Per Year

In [26]:
def annual_events(peak_date,first_year=1998,last_year=2026,type='calendar'):
    """
    Finds the number of blooms per year for the dataset.

    This function categorizes blooms into years based on their peak date. It uses the max_peaks function.

    Args:
        peak_date (list, required): List of peak chl dates for the dataset. No defaults.
        first_year (int, optional): First year in the time series of interest. Defaults to 1998.
        last_year (int, optional): Year after the last year of interest in the time series. Defaults to 2026

    Returns:
        Dictionary: A dictionary of the year and the number of events in that year.
    """
    #Year
    peak_years = []
    for date in peak_date:
        dt = pd.to_datetime(date)
        if type == 'calendar':
            peak_years.append(dt.year)
        elif type == 'biological':
            bio_year = dt.year if dt.month >= 7 else dt.year - 1
            peak_years.append(bio_year)
        else: 
            raise ValueError("Invalid year type. Choose 'calendar' or 'biological'.")
        
    blooms_per_year = Counter(peak_years)
    blooms_per_year = {year: blooms_per_year.get(year,0) for year in range (int(first_year),int(last_year))}
    return blooms_per_year

## Part 4: Bloom Characteristics Analysis

#### Duration of blooms

In [27]:
def bloom_duration(bloom_index,start_DOY,end_DOY,dataset,thld_start,thld_end):
    """
    Finds the length of each bloom event.

    Given the bloom index (1 to the length of start_DOY), the start DOY is subtracted from the end DOY to get the total duration of the bloom.
    It does this for the total bloom duration and the duration above the threshold.

    Args: 
        bloom_index : Bloom index of interest. Values range from 1 to len(start_DOY) + 1. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        thld_start (list, required): Pre-calculated DOY list of days where the chl first crosses the threshold.
        thld_end (list, required): Pre-calculated DOY list of days where the chl last dipped below the threshold

    Returns:
        tuple. A tuple of integers for the number of days of the bloom and the number of days above the threshold.
    """
    chl_median = dataset['smoothed_sg_win_15_poly_3'].interpolate_na(dim='time',method='linear')

    # STEP 1: Find the duration of the total bloom
    start_date = start_DOY[bloom_index]
    end_date = end_DOY[bloom_index]
    duration = end_date-start_date

    # STEP 2: Find the duration above the threshold for the bloom
    if thld_start[bloom_index] is not None:
        if bloom_index < len(thld_end):
            thld_end_idx = thld_end[bloom_index]
        else:
            thld_end_idx = len(chl_median)-1
        bloom_duration_thld = thld_end_idx-thld_start[bloom_index]
    else:
        bloom_duration_thld = "N/A"

    return duration, bloom_duration_thld

#### Integrated chlorophyll a

Using the raw data

Find the amount of chlorophyll under each bloom from 0 to peak chl-a value

In [28]:
def event_integrated_chla(raw_time,raw_chl,bloom_index,start_DOY,end_DOY):
    """
    Calculates the integrated chlorophyll-a concentration per bloom.

    This function uses trapezoid integration to estimate the amount of chlorophyll-a per bloom.
    It includes all chl-a values (0 to maximum value for the peak).
    The bounds of integration are determined by the start and end date found from the bloom_timing function.

    Args:
        raw_time (numpy.ndarray): The array of the raw time values for the dataset. No defaults.
        raw_chl (numpy.ndarray): The array of the raw chl values for the dataset. No defaults
        bloom_index (int, required): The bloom number for that index. In range of 0 - len(start_DOY). No defaults 
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. No defaults
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. No defaults
    
    Returns:
        Float. Integrated chlorophyll-a value for the bloom. 
    """
    bloom_start_time = np.asarray(start_DOY[bloom_index]).flatten().astype(int)
    bloom_end_time = np.asarray(end_DOY[bloom_index]).flatten().astype(int)

    lower_bound = int(bloom_start_time[0])
    upper_bound = int(bloom_end_time[0])
    bounded_time = raw_time[lower_bound:upper_bound]
    bounded_time = (bounded_time - bounded_time[0])/np.timedelta64(1,'D')
    bounded_chl = raw_chl[lower_bound:upper_bound]

    integrated_chl = scipy.integrate.trapezoid(bounded_chl,bounded_time,axis=0)
    return integrated_chl

In [29]:
def annual_integrated_chl(raw_time,raw_chl,year,start_DOY,end_DOY,peak_dates,type="calendar"):
    """
    Calculates the integrated chlorophyll for a year, considering only the chlorophyll during bloom events and the whole year.

    This function first calculates the total integrated chlorophyll for the whole year.
    Then it isolates the yearly chlorophyll into only the chlorophyll during bloom events.
    It then integrates over the full year to get the total integrated chlorophyll in mg/m^3 * days for only bloom periods.
    You can choose if the year runs from Jan 1 - Dec 31 (calendar) or July 1 to June 30 (biological).

    Args:
        raw_time (numpy.ndarray): The array of the raw time values for the dataset. No defaults.
        raw_chl (numpy.ndarray): The array of the raw chl values for the dataset. No defaults
        year (int, required): Year of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated. No defaults
        end_DOY (list, required): List of end DOYs pre-calculated. No defaults
        peak_dates (list, required): List of peak dates pre=calculated. No defaults
        type (str, optional): Sets up the type of year you are integrating over. Accepts "calendar" (Jan - Dec), "biological" (July - June), "bloom" (start of first bloom to end of last bloom for the year), or "bloom_bio" (biological bloom year). Defaults to "calendar"

    Returns:
        tuple. A tuple containing two float values where the first is the total integrated chlorophyll for the year and the second is only considering the bloom periods.
    """
    if type == "calendar":
        start_time = np.datetime64(f"{year}-01-01")
        end_time = np.datetime64(f"{year}-12-31")
    elif type == "biological":
        start_time = np.datetime64(f"{year}-07-01")
        end_time = np.datetime64(f"{year+1}-06-30")
    elif type == "bloom" or type == 'bloom_bio':
        # IDentify all blooms that initiated in this bloom year to find the dynamic start dates
        bloom_times = []
        for s_idx, e_idx, p_date in zip(start_DOY, end_DOY, peak_dates):
            b_start = int(np.ravel(s_idx)[0])
            b_end = int(np.ravel(e_idx)[0]) + 1
            b_time = raw_time[b_start:b_end]
            peak_time = np.datetime64(np.ravel(p_date)[0])
            peak_dt = pd.to_datetime(peak_time)
            if type == 'bloom_bio':
                b_year = peak_dt.tear if peak_dt.month >= 7 else peak_dt.year - 1
            else:
                b_year = peak_dt.year
            if b_year == year:
                bloom_times.append(b_time)
        if not bloom_times:
            return 0.0, 0.0
        start_time = min(t[0] for t in bloom_times)
        end_time = max(t[-1] for t in bloom_times)

    else:
        raise ValueError("Invalid year type. Choose 'calendar', 'biological', 'bloom', 'bloom_bio'.")

    # STEP 1: Total yearly chlorophyll
    mask = (raw_time >= start_time) & (raw_time <= end_time)
    time_year = raw_time[mask]
    chl_year = raw_chl[mask]

    bounded_time = (time_year - start_time)/np.timedelta64(1,'D')

    annual_integrated_chl = scipy.integrate.trapezoid(chl_year,bounded_time,axis=0)

    # STEP 2: Yearly integrated chlorophyll only including times of bloom 
    total_yearly_chl_during_bloom = 0.0
    for s_idx, e_idx, p_date in zip(start_DOY, end_DOY, peak_dates):
        bloom_start_time = int(np.ravel(s_idx)[0])
        bloom_end_time = int(np.ravel(e_idx)[0])+1

        bloom_time = raw_time[bloom_start_time:bloom_end_time]
        bloom_chl = raw_chl[bloom_start_time:bloom_end_time]
        peak_time = np.datetime64(np.ravel(p_date)[0])
        peak_dt = pd.to_datetime(peak_time)
        
        if type == "bloom" or type == 'bloom_bio':
            if type == 'bloom_bio':
                bloom_year = peak_dt.year if peak_dt.month >= 7 else peak_dt.year - 1
            else:
                bloom_year = peak_dt.year
            if bloom_year != year:
                continue
            bloom_bounded_time = bloom_time
            bloom_bounded_chl = bloom_chl
        else:
            year_mask = (bloom_time >= start_time)&(bloom_time<=end_time)
            if not year_mask.any():
                continue
        
            bloom_bounded_time = bloom_time[year_mask]
            bloom_bounded_chl = bloom_chl[year_mask]
        
        if len(bloom_bounded_time)>1:
            bloom_bounded_time = (bloom_bounded_time - bloom_bounded_time[0])/np.timedelta64(1,'D')
            bloom_integrated_chl = scipy.integrate.trapezoid(bloom_bounded_chl,bloom_bounded_time,axis=0)
            if not np.isnan(bloom_integrated_chl):
                total_yearly_chl_during_bloom += bloom_integrated_chl

    return annual_integrated_chl, total_yearly_chl_during_bloom

In [30]:
def monthly_integrated_chl(dataset,year,month):
    """
    Calculates the monthly integrated chl-a concentration (mg/m^3 * day)

    This function finds the total integrated chl-a concentration from the first day to the last day of the month of the year in question.

    Args:
        dataset (xarray.Dataset, required): Raw dataset (not smoothed) for analysis. No defaults
        year (int, required): The year for calculating the integrated chlorophyll. No defaults
        month (int, required): The month number from 1-12.

    Returns:
        Float. The total integrated chlorophyll for the year as one number. 
    """
    month_string = f"{month:02d}"
    time_slice = f"{year}-{month_string}"
    raw_data = dataset.sel(time=time_slice)
    if raw_data['time'].size <= 1:
        return 0.0
    chl = raw_data['CHL_bloom_only_nan'].values
    if np.isnan(chl).all():
        return 0.0
    time = raw_data['time'].values
    start_date = np.datetime64(f"{year}-{month_string}-01")

    bounded_time = (time - start_date)/np.timedelta64(1,'D')

    valid_mask = ~np.isnan(chl)
    valid_chl = chl[valid_mask]
    valid_time = bounded_time[valid_mask]

    if len(valid_chl) <= 1:
        return 0.0

    integrated_chl = scipy.integrate.trapezoid(valid_chl,valid_time,axis=0)
    return integrated_chl

In [31]:
def percent_annual_integrated_chl(dataset,year,bloom_index,start_DOY,end_DOY):
    """
    Calculates the percentage of the annual integrated chlorophyll that one bloom makes up.
    This function uses the yearly_integrated_chl and event_integrated_chla functions.

    Args: 
        dataset (xaray.Dataset,required): Raw dataset for analysis. No defaults
        year (int, required): The year for calculating total integrated chlorophyll. No defaults
        bloom_index (int, required): The index of the bloom for calculation. No defaults.
        start_DOY (list, required): The pre-calculated list of initiation DOYs. No defaults
        end_DOY (list, required): The pre-calculated list of termination DOYs. No defaults
    
    Returns:
        tuple. A tuple of two float values. The first is the percentge of the total annual chlorophyll and the second is the percentage of the annual chlorophyll in bloom periods.
    """
    total_annual_chl, bloom_period_integrated_chl = annual_integrated_chl(dataset,year,start_DOY=start_DOY,end_DOY=end_DOY)
    bloom_chl = event_integrated_chla(dataset,bloom_index,start_DOY=start_DOY,end_DOY=end_DOY)
    percent_of_total_annual = (bloom_chl/total_annual_chl)
    percent_bloom_period_annual = (bloom_chl/bloom_period_integrated_chl)
    return percent_of_total_annual, percent_bloom_period_annual

In [32]:
def bloom_metrics(clipped_thld,clipped_med,dataset,verbose=False):
    """
    Finds bloom metrics for each bloom in the dataset.

    This function computes the following metrics and returns them as a dataframe with the following labels:
        - Year: The year the bloom event peaked (int)
        - Bloom_classification: The classification (spring, fall, winter, other) (str)
        - Start_date: The date the bloom began based on the rate of change (str)
        - Peak_date: The date the bloom peaked based on the maximum chlorophyll concentration (str)
        - End_date: The date the bloom ended based on the rate of change (str)
        - Start_DOY: The DOY (1-366) that the bloom initiated (int)
        - End_DOY: The DOY (1-366) that the bloom terminated (int)
        - Peak_DOY: The DOY (1-366) that the bloom peaked (int)
        - Total_duration: The total number of days that bloom lasted (int)
        - First_exceed_date: The date the bloom first exceeded the threshold (str)
        - last_drop_date: The date the bloom last dropped below the threshold (str)
        - First_exceed_DOY: The DOY (1-366) that the bloom first exceeded the threshold (int)
        - last_drop_DOY: The DOY (1-366) that the bloom last dropped below the threshold (int)
        - Duration_above_threshold: The number of days a bloom spent above the threshold (int)
        - Bloom_Integrated_Chlorophyll: The total integrated chlorophyll concentration for the event (float)
        - Maximum_chlorophyll: The maximum chlorophyll concentration for the event (float)
        - Percent_Annual_Integrated_Chlorophyll: The percent of the annual integrated chlorophyll made up by the bloom (float)
        - Percent_Bloom_Period_Annual_Integrated_Chlorophyll: The percent of the bloom period only annual integrated chlorophyll made up by the bloom (float)

    Args:
        clipped_thld (float, required): The pre-calculated climatological threshold for the area. No defaults
        clipped_med (float, required): The pre-calculated climatological median for the area. No defaults
        dataset (xarray.Dataset): The dataset for analysis. No defaults
        verbose (Boolean, optional): Shows the print statements for checking if variables were successfully computed if True. Defaults to False
    Returns:
        pandas.DataFrame. A pandas dataframe of the bloom metrics for the region.
    """
    # STEP 1: Run bloom_timing function for bloom timing metrics
    start_DOY,end_DOY,bloom_events,peak_dates,peak_DOY,max_chl,_, thld_start,thld_end = bloom_timing(clipped_thld=clipped_thld,clipped_med=clipped_med,dataset=dataset)
    if verbose is True:
        print("Successfully found bloom timing metrics")

    # STEP 2: Create a bloom ID for each event
    bloom_indices = []
    for i in range(len(start_DOY)):
        bloom_index = i+1
        bloom_indices.append(bloom_index)
    if verbose is True:
        print("Successfully created bloom indices")

    #STEP 3: Identify bloom year based on peak dates
    peak_year = [date.year for date in peak_dates]
    bio_year = []
    for date in peak_dates:
        dt = pd.to_datetime(date)
        peak_bio_year = dt.year if dt.month >= 7 else dt.year - 1
        bio_year.append(peak_bio_year)
    if verbose is True:
        print("Successfully found bloom years")

    # STEP 4: Convert DOY values from bloom_timing function into dates and 1-366 DOY values.
    start_date_str = []
    start_365_DOY = []
    end_date_str = []
    end_365_DOY = []
    for i in range(len(start_DOY)):
        start_date_with_time = pd.to_datetime(dataset['time'].values[int(start_DOY[i])])
        start_date = start_date_with_time.date()
        start_DOY_365 = start_date_with_time.dayofyear
        if i<len(end_DOY):
            end_date_with_time = pd.to_datetime(dataset['time'].values[int(end_DOY[i])])
            end_date = end_date_with_time.date()
            end_DOY_365 = end_date_with_time.dayofyear
        else:
            end_date = "N/A"
            end_DOY_365 = "N/A"
        start_date_str.append(start_date)
        start_365_DOY.append(start_DOY_365)
        end_date_str.append(end_date)
        end_365_DOY.append(end_DOY_365)
    if verbose is True:
        print("Successfully converted bloom timing outputs to dates and DOYs")
    # STEP 5: Classify blooms as winter, spring, fall, or other
    bloom_class = bloom_classification(dataset=dataset,bloom_events=bloom_events)[0]
    if verbose is True:
        print("Successfully classified blooms")

    # STEP 6: Find dates where chl first and last crossed threshold
    start_date_thld = []
    start_thld_365_DOY = []
    end_date_thld = []
    end_thld_365_DOY = []
    for i in range(len(thld_start)):
        thld_start_date_time = pd.to_datetime(dataset['time'].values[int(thld_start[i])])
        thld_start_date = thld_start_date_time.date()
        thld_start_DOY_365 = thld_start_date_time.dayofyear
        if i<len(thld_end):
            thld_end_date_with_time = pd.to_datetime(dataset['time'].values[int(thld_end[i])])
            thld_end_date = thld_end_date_with_time.date()
            thld_end_DOY_365 = thld_end_date_with_time.dayofyear
        else:
            thld_end_date = "N/A"
            thld_end_DOY_365 = "N/A"
        start_date_thld.append(thld_start_date)
        start_thld_365_DOY.append(thld_start_DOY_365)
        end_date_thld.append(thld_end_date)
        end_thld_365_DOY.append(thld_end_DOY_365)
    if verbose is True:
        print("Successfully found threshold based dates and DOYs")

    # STEP 7: Find the total duration of each bloom 
    bloom_total_duration = []
    bloom_thld_duration = []
    for i in range(len(start_DOY)):
        bloom_length, bloom_length_thld = bloom_duration(i,start_DOY=start_DOY,end_DOY=end_DOY,dataset=dataset,thld_start=thld_start,thld_end=thld_end)
        bloom_total_duration.append(bloom_length)
        bloom_thld_duration.append(bloom_length_thld)
    if verbose is True:
        print("Successfully found bloom durations")
    
    # STEP 8: Integrated chlorophyll statistics
    full_chl_array = dataset['CHL_median'].values
    full_time_array = dataset['time'].values
    annual_cache = {}
    bio_cache = {}
    event_chl = []
    percent_chl = []
    percent_bloom_period_annual = []
    percent_bio_chl = []
    percent_bloom_period_bio = []
    for i in range(len(start_DOY)):
        year = peak_dates[i].year
        if year not in annual_cache:
            annual_cache[year] = annual_integrated_chl(raw_chl=full_chl_array,raw_time=full_time_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates)
        total_annual, bloom_period_annual = annual_cache[year]
        bloom_chl = event_integrated_chla(raw_time=full_time_array,raw_chl=full_chl_array,bloom_index=i,start_DOY=start_DOY,end_DOY=end_DOY)
        percent_annual = (bloom_chl/total_annual) if total_annual > 0 else 0
        percent_bloom_period = (bloom_chl/bloom_period_annual) if bloom_period_annual > 0 else 0
        percent_chl.append(percent_annual)
        percent_bloom_period_annual.append(percent_bloom_period)
        event_chl.append(bloom_chl)
        if year not in bio_cache:
            bio_cache[year] = annual_integrated_chl(raw_chl=full_chl_array,raw_time=full_time_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type='biological')
        total_bio_annual, bloom_period_bio = bio_cache[year]
        percent_bio_annual = (bloom_chl/total_bio_annual) if total_bio_annual > 0 else 0
        percent_bio_chl.append(percent_bio_annual)
        percent_bio_bloom_annual = (bloom_chl/bloom_period_bio) if bloom_period_bio > 0 else 0
        percent_bloom_period_bio.append(percent_bio_bloom_annual)
    if verbose is True:
        print("Successfully integrated chlorophyll")
    dataframe_data = {
        "Year": peak_year,
        "Biological Year": bio_year,
        "Bloom_classification": bloom_class,
        "Start_date": start_date_str,
        "Peak_date": peak_dates,
        "End_date": end_date_str,
        "Start_DOY": start_365_DOY,
        "End_DOY": end_365_DOY,
        "Peak_DOY": peak_DOY,
        "Total_duration": bloom_total_duration,
        "First_exceed_date": start_date_thld,
        "last_drop_date": end_date_thld,
        "First_exceed_DOY": start_thld_365_DOY,
        "last_drop_DOY": end_thld_365_DOY,
        "Duration_above_threshold": bloom_thld_duration,
        "Bloom_Integrated_Chlorophyll": event_chl,
        "Maximum_chlorophyll": max_chl,
        "Percent_Annual_Integrated_Chlorophyll": percent_chl,
        "Percent_Bloom_Period_Annual_Integrated_Chlorophyll": percent_bloom_period_annual,
        "Percent_Bio_Year_Integrated_Chlorophyll": percent_bio_chl,
        "Percent_Bio_Year_Bloom_Only_Chlorophyll": percent_bloom_period_bio
    }
    bloom_df = pd.DataFrame(dataframe_data,index=bloom_indices)
    bloom_df.index.name = "Bloom ID"
    return bloom_df

In [33]:
def summary_bloom_metrics(clipped_thld,clipped_med,dataset,verbose=False):
    """
    Finds summary (yearly and monthly) metrics for the region.

    This function finds the following metrics and returns as dataframe with the following labels:
    - Number_of_blooms: The number of blooms in that year based on the peak date (int)
    - Total_bloom_days: The total number of bloom days in that year (Jan 1 - Dec 31) (int)
    - Bloom_days_above_threshold: The number of days that year spent above the threshold during a bloom period (int)
    - Total_integrated_chl: The total integrated chlorophyll for the year (Jan 1 - Dec 31) (float)
    - Bloom_period_integrated_chl: The total integrated chlorophyll for the year (Jan 1 - Dec 31) during bloom periods only (float)
    - Biological_total_integrated: The total integrated chlorophyll for the biological year (July 1 - June 30) (float)
    - Biological_bloom_period_integrated: The total integrated chlorophyll for the biological year (July 1 - June 30) during bloom periods only (float)
    - January_integrated_chl: The integrated chlorophyll for January based on bloom periods only
    - February_integrated_chl: The integrated chlorophyll for February based on bloom periods only
    - March_integrated_chl: The integrated chlorophyll for March based on bloom periods only
    - April_integrated_chl: The integrated chlorophyll for April based on bloom periods only
    - May_integrated_chl: The integrated chlorophyll for May based on bloom periods only
    - June_integrated_chl: The integrated chlorophyll for June based on bloom periods only
    - July_integrated_chl: The integrated chlorophyll for July based on bloom periods only
    - August_integrated_chl: The integrated chlorophyll for August based on bloom periods only
    - September_integrated_chl: The integrated chlorophyll for September based on bloom periods only
    - October_integrated_chl: The integrated chlorophyll for October based on bloom periods only
    - November_integrated_chl: The integrated chlorophyll for November based on bloom periods only
    - December_integrated_chl: The integrated chlorophyll for December based on bloom periods only

    Args:
        clipped_thld (float, required): The pre-calculated climatological threshold for the area. No defaults
        clipped_med (float, required): The pre-calculated climatological median for the area. No defaults
        dataset (xarray.Dataset): The dataset for analysis. No defaults
        verbose (Boolean, optional): Shows the print statements for checking if variables were successfully computed if True. Defaults to False
    Returns:
        pandas.DataFrame. A pandas dataframe of the summary bloom metrics for the region.
    """
    # STEP 1: Run bloom_timing function for bloom timing metrics
    start_DOY,end_DOY,_,peak_dates,_,_,_,thld_start,thld_end = bloom_timing(clipped_thld=clipped_thld,clipped_med=clipped_med,dataset=dataset)
    if verbose is True:
        print("Successfully found bloom timing metrics")

    # STEP 2: Find the number of events per year (and the years)
    bloom_events_annual = annual_events(peak_date=peak_dates)
    bloom_events_bio = annual_events(peak_date=peak_dates,type='biological')
    if verbose is True:
        print("Successfully calculated the number of events per year")

    # STEP 3: Find the total number of bloom days and days above the threshold per year.
    bloom_days_annual, bloom_days_annual_above_thld = [],[]
    bio_bloom_days_annual, bio_bloom_days_above_thld = [], []
    for year in range(1998,2026):
        annual, bloom_period = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
        if verbose is True:
            print(f"Year: {year} | Total Bloom Days: {annual} | Days Above Threshold: {bloom_period}")
        bloom_days_annual.append(annual)
        bloom_days_annual_above_thld.append(bloom_period)
        annual_bio, bloom_period_bio = bloom_days_per_year(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end,type = 'biological')
        bio_bloom_days_annual.append(annual_bio)
        bio_bloom_days_above_thld.append(bloom_period_bio)
    if verbose is True:
        print("Successfully found the annual bloom days and days above the threshold")

    # STEP 4: Find the percentage of total bloom days that are above the threshold
    percent_annual_thld = []
    for year in range(1998,2026):
        percent = percent_bloom_days(dataset=dataset,year=year,start_DOY=start_DOY,end_DOY=end_DOY,thld_start=thld_start,thld_end=thld_end)
        percent_annual_thld.append(percent)
    percent_bio_thld = []
    for i in range(len(bio_bloom_days_annual)):
        if bio_bloom_days_annual[i] > 0:
            percent_bio = bio_bloom_days_above_thld[i]/bio_bloom_days_annual[i]
            percent_bio_thld.append(percent_bio)
        else:
            percent_bio_thld.append(0)
    if verbose is True:
        print("Successfully found the percentage of total bloom days above the threshold")

    # STEP 5: Find the annual integrated chlorophyll (total and bloom period only)
    full_chl_array = dataset['CHL_median'].values
    full_time_array = dataset['time'].values
    integrated_annual = []
    bloom_integrated_annual = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates)
        integrated_annual.append(annual_int)
        bloom_integrated_annual.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated the integrated chlorophyll for the calendar year")
    
    # STEP 6: Find the biological (July - June) year integrated chlorophyll
    biological_integrated_annual = []
    biological_bloom_int_annual = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type="biological")
        biological_integrated_annual.append(annual_int)
        biological_bloom_int_annual.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated integrated chlorophyll for biological year")

    # STEP 7: Find the bloom year (start of first bloom, end of last bloom) integrated chlorophyll
    bloom_year_integrated = []
    bloom_year_bloom_only = []
    for year in range (1998,2026):
        annual_int, bloom_period_int = annual_integrated_chl(raw_time=full_time_array,raw_chl=full_chl_array,year=year,start_DOY=start_DOY,end_DOY=end_DOY,peak_dates=peak_dates,type="bloom")
        bloom_year_integrated.append(annual_int)
        bloom_year_bloom_only.append(bloom_period_int)
    if verbose is True:
        print("Successfully calculated integrated chlorophyll for bloom year")

    # STEP 8: Monthly integrated chlorophyll per year (total and bloom period only)
    year = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
    month_options = ['January','February','March','April','May','June','July','August','September','October','November','December']
    jan_chl,feb_chl,mar_chl,apr_chl,may_chl,jun_chl,jul_chl,aug_chl,sep_chl,oct_chl,nov_chl,dec_chl = [],[],[],[],[],[],[],[],[],[],[],[]
    lists = [jan_chl,feb_chl,mar_chl,apr_chl,may_chl,jun_chl,jul_chl,aug_chl,sep_chl,oct_chl,nov_chl,dec_chl]
    for j in range(len(month_options)):
        month_actual = j+1
        monthly_chl = lists[j]
        for i in year:
            month_chl = monthly_integrated_chl(dataset,i,month_actual)
            monthly_chl.append(month_chl)
    if verbose is True:
        print("Successfully calculated the monthly integrated chlorophyll")

    data = {
        'Number_of_blooms': bloom_events_annual,
        'Number_of_blooms_biological': bloom_events_bio,
        'Total_bloom_days_calendar': bloom_days_annual,
        'Bloom_days_above_threshold_calendar': bloom_days_annual_above_thld,
        'Total_bloom_days_biological': bio_bloom_days_annual,
        'Bloom_days_above_threshold_biological': bio_bloom_days_above_thld,
        'Percent_total_bloom_days_above_threshold': percent_annual_thld,
        'Percent_total_bloom_days_above_threshold_biological': percent_bio_thld,
        'Total_integrated_chl': integrated_annual,
        'Bloom_period_integrated_chl': bloom_integrated_annual,
        'Biological_total_integrated': biological_integrated_annual,
        'Biological_bloom_period_integrated': biological_bloom_int_annual,
        'Bloom_year_integrated_chl': bloom_year_integrated,
        'Bloom_year_bloom_only_integrated': bloom_year_bloom_only,
        'January_integrated_chl': jan_chl,
        'February_integrated_chl': feb_chl,
        'March_integrated_chl': mar_chl,
        'April_integrated_chl': apr_chl,
        'May_integrated_chl': may_chl,
        'June_integrated_chl': jun_chl,
        'July_integrated_chl': jul_chl,
        'August_integrated_chl': aug_chl,
        'September_integrated_chl': sep_chl,
        'October_integrated_chl': oct_chl,
        'November_integrated_chl': nov_chl,
        'December_integrated_chl': dec_chl,
    }
    df = pd.DataFrame(data,index=year)
    df.index.name = "Year"
    if verbose is True:
        print("Successfully created dataframe")
    return df

In [ ]:
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
for x in range(5):
    df = summary_bloom_metrics(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_region[x])
    df.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Summary_Stats.csv',mode='w')
    print("Successfully saved file :)")

In [ ]:
data_region = [MABS,MABN,GB,GOMW,GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
clipped = [MABS_thld,MABN_thld,GB_thld,GOMW_thld,GOME_thld]
clipped_median = [MABS_median,MABN_median,GB_median,GOMW_median,GOME_median]
for x in range(5):
    df = bloom_metrics(clipped_thld=clipped[x],clipped_med=clipped_median[x],dataset=data_region[x])
    df.to_csv(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\{region_acro[x]}_Bloom_Metrics.csv',mode='w')
    print("Successfully saved file :)")

### Center of Gravity

In [56]:
daily_data.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
daily_data.rio.write_crs("epsg:4326", inplace=True)
clipped_daily_MABS = daily_data.rio.clip(MAB_south_loc.geometry.apply(mapping), shapefile.crs, drop=True)
clipped_daily_MABN = daily_data.rio.clip(MAB_north_loc.geometry.apply(mapping), shapefile.crs, drop=True)
clipped_daily_GB = daily_data.rio.clip(GB_whole_loc.geometry.apply(mapping), shapefile.crs, drop=True)
clipped_daily_GOMW = daily_data.rio.clip(GOM_west_loc.geometry.apply(mapping), shapefile.crs, drop=True)
clipped_daily_GOME = daily_data.rio.clip(GOM_east_loc.geometry.apply(mapping), shapefile.crs, drop=True)

In [ ]:
dataset = clipped_daily_MABS

threshold = MABS_thld
#threshold = threshold.squeeze("time",drop=True)
weights = dataset['CHL_median'].where(dataset['CHL_median'] > threshold).fillna(0)

total_chl = weights.sum(dim=['lat','lon'])
center_lon = (weights*dataset['lon']).sum(dim=['lat','lon'])/total_chl
center_lat = (weights*dataset['lat']).sum(dim=['lat','lon'])/total_chl

trends = pd.DataFrame({
    "lon":center_lon.values,
    "lat": center_lat.values,
    },
    index=dataset['time'].values
)
annual_df = trends.dropna().resample("YE").mean()

In [ ]:
bathym = cfeature.NaturalEarthFeature(name='bathymetry_K_200', scale='10m', category='physical')
bathym=unary_union(list(bathym.geometries()))
fig = plt.figure(figsize=(20,8))
map_projection = cartopy.crs.PlateCarree()
ax = plt.axes(projection=map_projection)

ax.set_extent([-77,-62,37,47])
ax.legend(fontsize=8,loc='lower right')

ax.add_feature(cartopy.feature.COASTLINE, linewidth=1)
ax.add_feature(cartopy.feature.LAND, zorder=100, facecolor='lightgrey')
ax.add_geometries(bathym, facecolor='none', edgecolor='black', crs=cartopy.crs.PlateCarree()) #Adding the shelf break line
states_provinces = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_1_states_provinces_lines',
    scale='50m',
    facecolor='none',
    edgecolor='gray',
    zorder = 150
)
ax.add_feature(states_provinces, linewidth=0.8)
years = annual_df.index.year
ax.plot(
    annual_df['lon'],
    annual_df['lat'],
    color='tab:red',
    linestyle = '-',
    linewidth=2,
    alpha=0.7,
    transform=map_projection,
    zorder=3
)
scatter = ax.scatter(
    annual_df['lon'],
    annual_df['lat'],
    c=years,
    cmap='viridis',
    s=130,
    edgecolor='black',
    transform=map_projection,
    zorder=4
)
ax.set_extent([-76,-74,37.5,38.5])
ax.gridlines(draw_labels=True)
ax.set_title('Chlorophyll a Climatological Median', fontsize=24)

### Plotting the heatmaps for each region

In [ ]:
data = [summary_MABS,summary_MABN,summary_GB,summary_GOMW,summary_GOME]
region_acro = ['MABS','MABN','GB','GOMW','GOME']
region_title = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
for x in range(5):
    data_set = data[x]
    data_for_hm = {
        'Year': data_set['Year'],
        'January': data_set['January Integrated chl'],
        'February': data_set['February Integrated chl'],
        'March': data_set['March Integrated chl'],
        'April': data_set['April Integrated chl'],
        'May': data_set['May Integrated chl'],
        'June': data_set['June Integrated chl'],
        'July': data_set['July Integrated chl'],
        'August': data_set['August Integrated chl'],
        'September': data_set['September Integrated chl'],
        'October': data_set['October Integrated chl'],
        'November': data_set['November Integrated chl'],
        'December': data_set['December Integrated chl'],
    }
    df = pd.DataFrame(data_for_hm)
    heatmap_data = df.set_index('Year') # Makes year the x axis

    plt.figure(figsize=(18,6))
    sns.heatmap(
        heatmap_data,
        cmap='Greens',
        annot=False,
        fmt=".1f",
        linewidths=0.5,
        cbar_kws={'label': "Integrated Chlorophyll ($mg/m^3 * day$)"}
    )
    plt.title(f"Monthly Integrated Chlorophyll by Year for {region_title[x]}")
    plt.xlabel("Year")
    plt.ylabel("month")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\{region_acro[x]}_heatmap_chl_intensity')

### Plotting the number of events per year per region (bar chart)

#### Single line bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(18,8))
width=0.14
x=summary_MABS['Year']
ax.bar(x-2*width,summary_MABS['Number_of_blooms'],color='gold',label='MAB South',edgecolor='black',width=width)
ax.bar(x-width,summary_MABN['Number_of_blooms'],color='cyan',label='MAB North',edgecolor='black',width=width)
ax.bar(x,summary_GB['Number_of_blooms'],color='darkorange',label='Georges Bank',edgecolor='black',width=width)
ax.bar(x+width,summary_GOMW['Number_of_blooms'],color='mediumorchid',label='GOM West',edgecolor='black',width=width)
ax.bar(x+2*width,summary_GOME['Number_of_blooms'],color='dodgerblue',label='GOM East',edgecolor='black',width=width)
ax.set_xticks(x)
tick_positions = [0,1,2,3,4,5,6]
tick_labels = ['0','1','2','3','4','5','6']
ax.set_yticks(tick_positions,labels=tick_labels)
ax.legend(fontsize=14)
ax.set_title("Annual Number of Blooms",fontsize=20)
ax.set_ylabel("Number of blooms per year",fontsize=14)
ax.set_xlabel("Year",fontsize=14)
ax.set_xlim(1997,2026)
ax.set_axisbelow(True)
ax.grid(axis='y',linestyle='-',alpha=0.7,color='gray')
plt.tight_layout()
plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_Blooms_Bars',dpi=300,bbox_inches='tight')

#### Decade(ish) bar chart (2 rows)

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=1,figsize=(16,8),sharey=True)
decades = [
    (1998, 2010, axes[0], "1998 - 2010"),
    (2011, 2025, axes[1], "2011 - 2025")
]
width = 0.15
for start, end, ax, title in decades:
    mabs_bounded = summary_MABS[(summary_MABS['Year'] >= start) & (summary_MABS['Year'] <= end)]
    mabn_bounded = summary_MABN[(summary_MABN['Year'] >= start) & (summary_MABN['Year'] <= end)]
    gb_bounded = summary_GB[(summary_GB['Year'] >= start) & (summary_GB['Year'] <= end)]
    gomw_bounded = summary_GOMW[(summary_GOMW['Year'] >= start) & (summary_GOMW['Year'] <= end)]
    gome_bounded = summary_GOME[(summary_GOME['Year'] >= start) & (summary_GOME['Year'] <= end)]

    x=mabs_bounded['Year']
    ax.bar(x-2*width,mabs_bounded['Number_of_blooms'],color='gold',label='MAB South',edgecolor='black',width=width)
    ax.bar(x-width,mabn_bounded['Number_of_blooms'],color='cyan',label='MAB North',edgecolor='black',width=width)
    ax.bar(x,gb_bounded['Number_of_blooms'],color='darkorange',label='Georges Bank',edgecolor='black',width=width)
    ax.bar(x+width,gomw_bounded['Number_of_blooms'],color='mediumorchid',label='GOM West',edgecolor='black',width=width)
    ax.bar(x+2*width,gome_bounded['Number_of_blooms'],color='dodgerblue',label='GOM East',edgecolor='black',width=width)
    ax.set_xticks(x)
    tick_positions = [0,1,2,3,4,5,6]
    tick_labels = ['0','1','2','3','4','5','6']
    ax.set_yticks(tick_positions,labels=tick_labels)
    ax.set_ylabel("Blooms per year",fontsize=14)
    ax.set_xlabel("Year",fontsize=14)
    ax.set_xlim(start-0.5,end+0.5)
    ax.set_axisbelow(True)
    ax.grid(axis='y',linestyle='-',alpha=0.5,color='gray')
axes[0].legend(fontsize=14,loc='upper left')
axes[1].set_xlabel('Year', fontsize=14)
plt.tight_layout()

#### Heatmap

In [42]:
data = {
    'Year': summary_MABS['Year'],
    'MAB South': summary_MABS['Number_of_blooms_biological'],
    'MAB North': summary_MABN['Number_of_blooms_biological'],
    'Georges Bank': summary_GB['Number_of_blooms_biological'],
    'GOM West': summary_GOMW['Number_of_blooms_biological'],
    'GOM East': summary_GOME['Number_of_blooms_biological']
}
annual_bloom_events_heatmap = pd.DataFrame(data)
annual_bloom_events_heatmap = annual_bloom_events_heatmap.sort_values('Year')
annual_bloom_events_heatmap = annual_bloom_events_heatmap.set_index('Year').T

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
x_labels = range(1998,2026)
sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(x_labels)]
num_segments = 7
original_cmap = cmocean.cm.thermal
segmented_cmap = original_cmap.resampled(num_segments)
ax = sns.heatmap(annual_bloom_events_heatmap,
            cmap=segmented_cmap,
            annot=True,
            fmt="d",
            linewidth=0.5,
            vmin = 0,
            vmax=6,
            cbar_kws={'label':'Number of blooms per year'}
            )
cbar = ax.collections[0].colorbar
boundaries = np.linspace(0, 6, num_segments+1)
segment_centers = (boundaries[:-1] + boundaries[1:])/2
cbar.set_ticks(segment_centers)
tick_labels = ['0','1','2','3','4','5','6']
cbar.set_ticklabels(tick_labels)
ax.set_title("Annual Bloom Events",fontsize=20)
ax.set_xlabel("Biological Year",fontsize=14)
ax.set_ylabel("Region",fontsize=14)
ax.set_xticklabels(sparse_labels)
plt.xticks(rotation=75)
plt.tight_layout()
plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_Blooms_Heatmap',dpi=300,bbox_inches='tight')

### Bloom days per year heatmap

In [194]:
data = {
    'Year': summary_MABS['Year'],
    'MAB South': summary_MABS['Percent_total_bloom_days_above_threshold_biological'],
    'MAB North': summary_MABN['Percent_total_bloom_days_above_threshold_biological'],
    'Georges Bank': summary_GB['Percent_total_bloom_days_above_threshold_biological'],
    'GOM West': summary_GOMW['Percent_total_bloom_days_above_threshold_biological'],
    'GOM East': summary_GOME['Percent_total_bloom_days_above_threshold_biological']
}
df = pd.DataFrame(data)
df = df.sort_values('Year')
bloom_days_heatmap = df.set_index('Year').T

In [ ]:
x_labels = ['1998/1999','1999/2000','2000/2001','2001/2002','2002/2003','2003/2004','2004/2005','2005/2006','2006/2007','2007/2008','2008/2009','2009/2010','2010/2011','2011/2012','2012/2013','2013/2014','2014/2015','2015/2016','2016/2017','2017/2018','2018/2019','2019/2020','2020/2021','2021/2022','2022/2023','2023/2024','2024/2025','2025/2026']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(bloom_days_heatmap,
            cmap=cmocean.cm.thermal,
            annot=True,
            fmt=".2f",
            linewidth=0.5,
            cbar_kws={'label':'Number'}
            )
ax.set_title("Annual Bloom Days Above the Threshold",fontsize=20)
ax.set_xlabel("Year",fontsize=14)
ax.set_ylabel("Region",fontsize=14)
ax.set_xticklabels(sparse_labels)
plt.xticks(rotation=75)
plt.tight_layout()
#plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Bio_Annual_Bloom_Days_Threshold_Heatmap',dpi=300,bbox_inches='tight')

### Plotting Gantt Charts for each region

In [ ]:
csv_data = [bloom_MABS,bloom_MABN,bloom_GB,bloom_GOMW,bloom_GOME]
titles = ["Middle Atlantic Bight South", "Middle Atlantic Bight North", "Georges Bank", "Gulf of Maine West", "Gulf of Maine East"]
for x in range(5):
    data = csv_data[x]
    data['Start date day']=pd.to_datetime(data['Start date'])
    data['End date day']=pd.to_datetime(data['End date'])
    data['Start date']=data['Start date day'].dt.strftime('%m-%d-%Y')
    data['End date']=data['End date day'].dt.strftime('%m-%d-%Y')
    data['Normalized Start'] = pd.to_datetime('2024-'+data['Start date day'].dt.strftime('%m-%d'))
    data['Normalized End'] = pd.to_datetime('2024-'+data['End date day'].dt.strftime('%m-%d'))

    cross_year = data['End date day'].dt.year>data['Start date day'].dt.year
    if cross_year.any():
        cross_blooms = data[cross_year].copy()
        data.loc[cross_year, 'Normalized End'] = pd.to_datetime('2025-'+data['End date day'].dt.strftime('%m-%d'))
        cross_blooms['Normalized Start'] = pd.to_datetime('2024-01-01')
        data = pd.concat([data,cross_blooms],ignore_index=True)
    fig = px.timeline(
        data,
        x_start="Normalized Start",
        x_end="Normalized End",
        y="Year",
        color = "Total duration (days)",
        title=f"Bloom Gantt Chart {titles[x]}",
        width = 750,
        height = 550,
        hover_data={
            "Start date":True,
            "End date":True,
            "Start date day": False,
            "End date day": False
        }
    )
    fig.update_layout(
        coloraxis_cmin=0,
        coloraxis_cmax = 350
    )
    fig.update_xaxes(
        dtick = "M1",
        tickformat="%b",
        ticklabelmode="period",
        range = ["2024-01-01","2024-12-31"]
    )
    years = sorted(data['Year'].unique())
    fig.update_yaxes(
        type='category',
        tickmode='array',
        tickvals=years,
        autorange = 'reversed'
        )
    fig.show()
    #fig.write_html(rf"C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\{titles[x]}_threshold_Gantt_Chart.html")

### Plotting heatmaps overlaid with Gantt charts for each region

### January - December Code

In [34]:
def heatmap_jan_dec(summary_data,individual_data,region_title,title_fontsize=20, axes=None, colorbars='on'):
    heatmap_x_dates = pd.to_datetime([f"2024-{str(month).zfill(2)}-01" for month in range(1,13)] + ['2025-01-01'])
    data_set = summary_data
    data_for_hm = {
        'Year': data_set['Year'],
        'January': data_set['January_integrated_chl'],
        'February': data_set['February_integrated_chl'],
        'March': data_set['March_integrated_chl'],
        'April': data_set['April_integrated_chl'],
        'May': data_set['May_integrated_chl'],
        'June': data_set['June_integrated_chl'],
        'July': data_set['July_integrated_chl'],
        'August': data_set['August_integrated_chl'],
        'September': data_set['September_integrated_chl'],
        'October': data_set['October_integrated_chl'],
        'November': data_set['November_integrated_chl'],
        'December': data_set['December_integrated_chl'],
    }
    df = pd.DataFrame(data_for_hm)
    df = df.sort_values('Year')
    heatmap_data = df.set_index('Year')

    individual_dataset = individual_data.copy()
    individual_dataset['Start date day']=pd.to_datetime(individual_dataset['Start_date'])
    individual_dataset['End date day']=pd.to_datetime(individual_dataset['End_date'])
    individual_dataset['Start_date']=individual_dataset['Start date day'].dt.strftime('%m-%d-%Y')
    individual_dataset['End_date']=individual_dataset['End date day'].dt.strftime('%m-%d-%Y')
    individual_dataset['End Year'] = individual_dataset['End date day'].dt.year

    new_dataset_rows = []

    for index, row in individual_dataset.iterrows():
        start_date = row['Start date day']
        end_date = row['End date day']
        start_year = start_date.year
        end_year = end_date.year

        current_year = start_year
        while current_year <= end_year:
            row_copy = row.copy()
            if current_year == start_year:
                row_start = start_date
            else:
                row_start = pd.Timestamp(year=current_year,month=1,day=1)
            if current_year == end_year:
                row_end = end_date
            else:
                row_end = pd.Timestamp(year=current_year,month=12,day=31,hour=23,minute=59,second=59)
            row_copy['End Year'] = current_year
            
            dummy_year = 2024
            row_copy['Normalized Start'] = pd.Timestamp(year=dummy_year,month=row_start.month,day=row_start.day)
            row_copy['Normalized End'] = pd.Timestamp(year=dummy_year,month=row_end.month,day=row_end.day)
            new_dataset_rows.append(row_copy)
            current_year = current_year+1

    individual_dataset = pd.DataFrame(new_dataset_rows).reset_index(drop=True)
    heatmap_data = heatmap_data[heatmap_data.index >= 1998]
    individual_dataset = individual_dataset[individual_dataset['End Year'] >= 1998]
    if axes is None:
        fig, ax = plt.subplots(figsize=(14,8), layout='tight')
    else:
        ax=axes
        fig = ax.get_figure()

    #Heatmap
    y_center = heatmap_data.index.astype(int).values
    y_edges = np.append(y_center - 0.5, y_center[-1] + 0.5)

    x_dates = pd.date_range(start='2024-01-01',end='2025-01-01',freq='MS')
    x_edges = mdates.date2num(x_dates)

    Z = heatmap_data.values.astype(float)
    Z[Z <= 0] = np.nan

    cmap_hm = plt.get_cmap('Greens').copy()
    cmap_hm.set_bad(color='white')
    hm = ax.pcolormesh(x_edges,y_edges,Z,cmap=cmap_hm,norm=mcolors.LogNorm(vmin=0.1,vmax=100), shading='flat')
    
    for date in heatmap_x_dates:
        ax.axvline(
            mdates.date2num(date),
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )

    for year in y_edges:
        ax.axhline(
            year,
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )
    lefts = mdates.date2num(individual_dataset['Normalized Start'])
    rights = mdates.date2num(individual_dataset['Normalized End'])
    widths = rights - lefts
    y_coords = individual_dataset['End Year'].astype(int)

    durations = pd.to_numeric(individual_dataset['Total_duration'], errors='coerce').fillna(0)
    norm = mcolors.Normalize(vmin=0, vmax =350)
    cmap_bars = plt.get_cmap('Wistia')
    bar_colors = cmap_bars(norm(durations))

    ax.barh(y_coords, widths, left=lefts, height =0.4, color=bar_colors, edgecolor='black', linewidth=1, zorder=5)
    #ax.set_title(f"Bloom Duration and Magnitude in {region_title}",fontsize=title_fontsize)
    ax.set_title("Calendar Year (January - December)", fontsize=18)
    ax.set_yticks(range(1998,2026))
    ax.invert_yaxis()

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(mdates.date2num(pd.Timestamp('2024-01-01')),mdates.date2num(pd.Timestamp('2025-01-01')))

    if colorbars == 'on':
        cax_hm = ax.inset_axes([1.02,0.0,0.03,1.0])
        cax_bars = ax.inset_axes([1.13,0.0,0.03,1.0])

        cbar_hm = fig.colorbar(hm, cax=cax_hm)
        cbar_hm.set_label("Integrated Chlorophyll ($mg/m^3 * days$)",fontsize=14)
        cbar_hm.ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

        sm = plt.cm.ScalarMappable(cmap=cmap_bars, norm=norm)
        sm.set_array([])
        cbar_bars = fig.colorbar(sm, cax=cax_bars)
        cbar_bars.set_label("Duration of event (days)",fontsize=14)
    return fig

#### July - June Code

In [35]:
def heatmap_jul_jun(summary_data,individual_data,region_title,title_fontsize=20, axes=None, colorbars='on'):
    heatmap_x_dates = pd.to_datetime([f"2023-{str(month).zfill(2)}-01" for month in range(7,13)] + [f"2024-{str(month).zfill(2)}-01" for month in range (1,7)])
    data_set = summary_data
    data_for_hm = {
        'Year': data_set['Year'],
        'July': data_set['July_integrated_chl'],
        'August': data_set['August_integrated_chl'],
        'September': data_set['September_integrated_chl'],
        'October': data_set['October_integrated_chl'],
        'November': data_set['November_integrated_chl'],
        'December': data_set['December_integrated_chl'],
        'January': data_set['January_integrated_chl'],
        'February': data_set['February_integrated_chl'],
        'March': data_set['March_integrated_chl'],
        'April': data_set['April_integrated_chl'],
        'May': data_set['May_integrated_chl'],
        'June': data_set['June_integrated_chl'],
    }
    df = pd.DataFrame(data_for_hm)
    df = df.sort_values('Year')
    spring_months = ['January','February','March','April','May','June']
    df[spring_months] = df[spring_months].shift(-1)
    heatmap_data = df.set_index('Year')

    individual_dataset_july_start = individual_data.copy()
    individual_dataset_july_start['Start date day']=pd.to_datetime(individual_dataset_july_start['Start_date'])
    individual_dataset_july_start['End date day']=pd.to_datetime(individual_dataset_july_start['End_date'])
    individual_dataset_july_start['Start_date']=individual_dataset_july_start['Start date day'].dt.strftime('%m-%d-%Y')
    individual_dataset_july_start['End_date']=individual_dataset_july_start['End date day'].dt.strftime('%m-%d-%Y')
    individual_dataset_july_start['End Year'] = individual_dataset_july_start['End date day'].dt.year

    new_dataset_rows = []

    for index, row in individual_dataset_july_start.iterrows():
        start_date = row['Start date day']
        end_date = row['End date day']
        if start_date.month < 7:
            start_year = start_date.year-1
        else:
            start_year = start_date.year
        if end_date.month < 7:
            end_year = end_date.year-1
        else:
            end_year = end_date.year

        current_year = start_year
        while current_year <= end_year:
            row_copy = row.copy()
            if current_year == start_year:
                row_start = start_date
            else:
                row_start = pd.Timestamp(year=current_year,month=7,day=1)
            if current_year == end_year:
                row_end = end_date
            else:
                row_end = pd.Timestamp(year=current_year+1,month=6,day=30,hour=23,minute=59,second=59)
            row_copy['End Year'] = current_year

            if row_start.month >= 7:
                dummy_start_year = 2023
            else:
                dummy_start_year = 2024
            if row_end.month >= 7:
                dummy_end_year = 2023
            else:
                dummy_end_year = 2024

            row_copy['Normalized Start'] = pd.Timestamp(year=dummy_start_year,month=row_start.month,day=row_start.day)
            row_copy['Normalized End'] = pd.Timestamp(year=dummy_end_year,month=row_end.month,day=row_end.day)
            new_dataset_rows.append(row_copy)
            current_year = current_year+1

    individual_dataset_july_start = pd.DataFrame(new_dataset_rows).reset_index(drop=True)
    heatmap_data = heatmap_data[heatmap_data.index >= 1998]
    individual_dataset_july_start = individual_dataset_july_start[individual_dataset_july_start['End Year'] >= 1998]
    if axes is None:
        fig, ax = plt.subplots(figsize=(14,8), layout='tight')
    else:
        ax = axes
        fig = ax.get_figure()

    #Heatmap
    y_center = heatmap_data.index.astype(int).values
    y_edges = np.append(y_center - 0.5, y_center[-1] + 0.5)

    x_dates = pd.date_range(start='2023-07-01',end='2024-07-01',freq='MS')
    x_edges = mdates.date2num(x_dates)

    Z = heatmap_data.values.astype(float)
    Z[Z <= 0] = np.nan

    cmap_hm = plt.get_cmap('Greens').copy()
    cmap_hm.set_bad(color='white')
    hm = ax.pcolormesh(x_edges,y_edges,Z,cmap=cmap_hm,norm=mcolors.LogNorm(vmin=0.1,vmax=100), shading='flat')
    
    for date in heatmap_x_dates:
        ax.axvline(
            mdates.date2num(date),
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )

    for year in y_edges:
        ax.axhline(
            year,
            color='darkgray',
            alpha=0.5,
            linewidth=1.0,
            zorder=2
        )
    lefts = mdates.date2num(individual_dataset_july_start['Normalized Start'])
    rights = mdates.date2num(individual_dataset_july_start['Normalized End'])
    widths = rights - lefts
    y_coords = individual_dataset_july_start['End Year'].astype(int)

    durations = pd.to_numeric(individual_dataset_july_start['Total_duration'], errors='coerce').fillna(0)
    norm = mcolors.Normalize(vmin=0, vmax =350)
    cmap_bars = plt.get_cmap('Wistia')
    bar_colors = cmap_bars(norm(durations))

    ax.barh(y_coords, widths, left=lefts, height =0.4, color=bar_colors, edgecolor='black', linewidth=1, zorder=5)
    ax.set_title(f"Bloom Duration and Magnitude in {str(region_title)}",fontsize=title_fontsize)
    #ax.set_title("Biological Year (July - June)",fontsize=18) 
    ax.set_yticks(range(1998,2026))
    ax.invert_yaxis()

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_xlim(mdates.date2num(pd.Timestamp('2023-07-01')),mdates.date2num(pd.Timestamp('2024-06-30')))

    ax.axvline(mdates.date2num(pd.Timestamp('2024-01-01')),color='aqua',linestyle='--',linewidth=2,alpha=0.7)
    if colorbars == 'on':
        cax_hm = ax.inset_axes([1.02,0.0,0.03,1.0])
        cax_bars = ax.inset_axes([1.15,0.0,0.03,1.0])

        cbar_hm = fig.colorbar(hm, cax=cax_hm)
        cbar_hm.set_label("Integrated Chlorophyll ($mg/m^3 * days$)",fontsize=12)
        cbar_hm.ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

        sm = plt.cm.ScalarMappable(cmap=cmap_bars, norm=norm)
        sm.set_array([])
        cbar_bars = fig.colorbar(sm, cax=cax_bars)
        cbar_bars.set_label("Duration of event (days)",fontsize=12)
    return fig

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=2,figsize=(18,8))
fig.subplots_adjust(wspace=0.1)
axes_flat = axes.flatten()
heatmap_jan = heatmap_jan_dec(summary_MABN,bloom_MABN,"Middle Atlantic Bight North",title_fontsize=18,axes = axes_flat[0],colorbars='off')
heatmap_jul = heatmap_jul_jun(summary_MABN,bloom_MABN,"Middle Atlantic Bight North",title_fontsize=18,axes=axes_flat[1])
fig.suptitle("Bloom Duration and Intensity in the Middle Atlantic Bight North",fontsize=20)
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Heatmaps\Heatmap_Comparison_MABN.png',dpi=300,bbox_inches='tight')

### Number of bloom classifications for each region

In [ ]:
fig, ax = plt.subplots(figsize=(18,8))
plt.grid(axis='y')
width=0.15
classifications = ['Spring', 'Fall', 'Winter', 'Summer']
x = np.arange(len(classifications))
def counts(df):
    counting = df['Bloom_classification'].value_counts()
    aligned_counts = counting.reindex(classifications,fill_value=0)
    return aligned_counts

counts_MABS = counts(bloom_MABS)
counts_MABN = counts(bloom_MABN)
counts_GB = counts(bloom_GB)
counts_GOMW = counts(bloom_GOMW)
counts_GOME = counts(bloom_GOME)

ax.bar(x-2*width,counts_MABS,color='gold',label='MAB South',edgecolor='black',width=0.15,zorder=20)
ax.bar(x-width,counts_MABN,color='cyan',label='MAB North',edgecolor='black',width=0.15,zorder=20)
ax.bar(x,counts_GB,color='darkorange',label='Georges Bank',edgecolor='black',width=0.15,zorder=20)
ax.bar(x+width,counts_GOMW,color='mediumorchid',label='GOM West',edgecolor='black',width=0.15,zorder=20)
ax.bar(x+2*width,counts_GOME,color='dodgerblue',label='GOM East',edgecolor='black',width=0.15,zorder=20)
ax.set_xticks(x)
ax.set_xticklabels(classifications)
ax.legend(fontsize=14)
ax.set_title("Classification of blooms in each region",fontsize=20)
ax.set_ylabel("Number of blooms in classification",fontsize=14)
ax.set_xlabel("Classification",fontsize=14)
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\bloom_classification.png',dpi=300,bbox_inches='tight')

### Metric Plots of Regions

In [36]:
def scatter_plot_data(dataset,interest_variable):
    """
    Finds the variable of interest for the spring and fall bloom.

    This function uses the metric csv file to find the variable of interest of the spring and fall blooms within a region.

    Args:
        dataset (pandas.DataFrame, required): The dataframe of all bloom metrics for the region. Must include 'Bloom ID', 'Year', and 'Bloom Classification' columns. No defaults.
        interest_variable (str, required): The variable of interest. The header in the dataframe. No defaults.
    Returns:
        tuple. A tuple containing (fall_x, fall_y, spring_x, spring_y), where:
            fall_x (list): The list of years where the fall bloom had that metric.
            fall_y (list): The list of values for the metric of interest for the fall bloom.
            spring_x (list): The list of years where the spring bloom had that metric.
            spring_y (list): The list of values for the metric of interest for the spring bloom.
    """
    fall_x = []
    fall_y = []
    spring_x = []
    spring_y = []
    variable_of_interest = str(interest_variable)

    target_values = dataset[variable_of_interest].tolist()
    years = dataset['Year'].tolist()
    bloom_classification = dataset['Bloom_classification'].tolist()
    start_DOYs = dataset['Start_DOY'].tolist()

    cross_boundary_variables = ['Peak_DOY', 'End_DOY', 'last_drop_DOY', 'First_exceed_DOY'] #Variables that might cross the year boundary

    for bloom_class, year, val, start_val in zip(bloom_classification, years, target_values, start_DOYs):
        if variable_of_interest in cross_boundary_variables:
            if not pd.isna(val) and not pd.isna(start_val) and val<start_val: #Checks for nan values and that the value is less than the start DOY value
                #Leap year verification
                is_leap_year = (year % 4 == 0 and year % 100 !=0) or (year % 400 == 0)
                val += 366 if is_leap_year else 365 #Adds 366 or 365 if the value is less than the start DOY
        if bloom_class == 'Fall':
            fall_y.append(val)
            fall_x.append(year)
        elif bloom_class == 'Spring':
            spring_y.append(val)
            spring_x.append(year)
    return fall_x, fall_y, spring_x, spring_y

In [65]:
region_acro = ['MABS','MABN','GB','GOMW','GOME']
data = [bloom_MABS,bloom_MABN,bloom_GB,bloom_GOMW,bloom_GOME]
color_options = ['gold','cyan','darkorange','mediumorchid','dodgerblue']
interest_variable= ['Start_DOY', 'Peak_DOY', 'End_DOY','Total_duration','First_exceed_DOY','last_drop_DOY','Duration_above_threshold','Bloom_Integrated_Chlorophyll','Maximum_chlorophyll']
variable_title= ['Initiation Day of Year', 'Peak Day of Year', 'Termination Day of Year','Duration','First Exceedance of Threshold (DOY)','Last Drop Below Threshold (DOY)','Duration Above Threshold','Bloom Integrated Chlorophyll','Maximum Chlorophyll']
titles = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
x_sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for j in range(len(interest_variable)):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True)
    axes_flat_fall = axes_fall.flatten()
    for i in range(5):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        axes_flat_fall[i].clear()
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]

        num_points_fall = len(clean_fall_x)
        has_enough_points_fall = num_points_fall >= 20
        sns.regplot(
            x=clean_fall_x,
            y=clean_fall_y,
            ax=axes_flat_fall[i],
            color=color_options[i],
            fit_reg=has_enough_points_fall
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
        if has_enough_points_fall:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_fall}\n(No trendline)"
        if p_value <= 0.0559:
            axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top'
        )
        else:
                axes_flat_fall[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[i].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[i].set_title(titles[i],fontsize=18)
        axes_flat_fall[i].set_xlabel('Year',fontsize=14)
        axes_flat_fall[i].set_ylabel(interest_variable[j],fontsize=14)
        axes_flat_fall[i].set_xticks(x_ticks)
        axes_flat_fall[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_fall[i].tick_params(labelleft=True)
    axes_flat_fall[5].set_visible(False)
    fig_fall.suptitle("Fall Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Fall_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9), sharey=True)
    axes_flat_spring = axes_spring.flatten()
    for i in range(5):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data[i],interest_variable=interest_variable[j])
        axes_flat_spring[i].clear()
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]

        num_points_spring = len(clean_spring_x)
        has_enough_points_spring = num_points_spring >= 20
        sns.regplot(
            x=clean_spring_x,
            y=clean_spring_y,
            ax=axes_flat_spring[i],
            color=color_options[i],
            fit_reg=has_enough_points_spring
        )
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[i].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

        if has_enough_points_spring:
            slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
            r_square = r_value**2
            stats_text = f"p-value = {p_value:.4f}"
        else:
            stats_text = f"N = {num_points_spring}\n(No trendline)"
        if p_value <=0.0559:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top'
            )
        else:
            axes_flat_spring[i].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[i].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[i].set_title(titles[i],fontsize=18)
        axes_flat_spring[i].set_xlabel('Year',fontsize=14)
        axes_flat_spring[i].set_ylabel(interest_variable[j],fontsize=14)
        axes_flat_spring[i].set_xticks(x_ticks)
        axes_flat_spring[i].set_xticklabels(x_sparse_labels,rotation=70)
        axes_flat_spring[i].tick_params(labelleft=True)
    axes_flat_spring[5].set_visible(False)
    fig_spring.suptitle("Spring Bloom " + variable_title[j] + " by Region",fontsize=20)
    filename = f"Spring_{interest_variable[j]}_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Metric_Scatter_Plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)

### Region Plots of Metrics

In [64]:
#titles = ['Gulf of Maine West','Gulf of Maine East']
#region_acro = ['GOMW','GOME']
#data = [bloom_GOMW,bloom_GOME]
#color_options = ['mediumorchid','dodgerblue']
region_acro = ['MABS','MABN','GB','GOMW','GOME']
data = [bloom_MABS,bloom_MABN,bloom_GB,bloom_GOMW,bloom_GOME]
color_options = ['gold','mediumturquoise','darkorange','mediumorchid','dodgerblue']
interest_variable= ['Start_DOY', 'Peak_DOY', 'End_DOY','Total_duration','Bloom_Integrated_Chlorophyll','Maximum_chlorophyll']
variable_title= ['Initiation Day of Year', 'Peak Day of Year', 'Termination Day of Year','Duration','Bloom Integrated Chlorophyll','Maximum Chlorophyll']
titles = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
for i in range(5):
    fig_fall, axes_fall = plt.subplots(nrows=2, ncols=3, figsize=(15,9))
    axes_flat_fall = axes_fall.flatten()
    data_set = data[i]
    #data_set = data_set[data_set['Year'] != 2023]
    for j in range(len(interest_variable)):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        sns.regplot(
            x=fall_x,
            y=fall_y,
            ax=axes_flat_fall[j],
            color=color_options[i]
        )
        fall_x = np.array(fall_x)
        fall_y = np.array(fall_y)
        mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
        clean_fall_x = fall_x[mask]
        clean_fall_y = fall_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <= 0.0559:
            axes_flat_fall[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[j].transAxes,
                fontsize=15,
                color = 'red',
                verticalalignment='top'
        )
        else:
                axes_flat_fall[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_fall[j].transAxes,
                fontsize=15,
                color = 'black',
                verticalalignment='top'
                )
        axes_flat_fall[j].set_title(variable_title[j],fontsize=18)
        axes_flat_fall[j].set_xlabel('Year',fontsize=14)
        axes_flat_fall[j].set_ylabel(interest_variable[j],fontsize=14)
        axes_flat_fall[j].set_xticks(x_ticks)
        axes_flat_fall[j].set_xticklabels(sparse_labels,rotation=70)
        if 'DOY' in interest_variable[j]:
            axes_flat_fall[j].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
    fig_fall.suptitle(f"{titles[i]} Fall Bloom Metrics",fontsize=20)
    filename = f"{region_acro[i]}_Fall_Bloom_Metrics_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_fall)

    fig_spring, axes_spring = plt.subplots(nrows=2, ncols=3, figsize=(15,9))
    axes_flat_spring = axes_spring.flatten()
    for j in range(len(interest_variable)):
        fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data_set,interest_variable=interest_variable[j])
        sns.regplot(
            x=spring_x,
            y=spring_y,
            ax=axes_flat_spring[j],
            color=color_options[i]
        )
        spring_x = np.array(spring_x)
        spring_y = np.array(spring_y)
        mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
        clean_spring_x = spring_x[mask]
        clean_spring_y = spring_y[mask]
        slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
        r_square = r_value**2
        stats_text = f"p-value = {p_value:.4f}"
        if p_value <=0.0559:
            axes_flat_spring[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[j].transAxes,
                fontsize=15,
                color='red',
                verticalalignment='top'
            )
        else:
            axes_flat_spring[j].text(
                0.05,0.95,
                stats_text,
                transform=axes_flat_spring[j].transAxes,
                fontsize=15,
                color='black',
                verticalalignment='top'
            )
        axes_flat_spring[j].set_title(variable_title[j],fontsize=18)
        axes_flat_spring[j].set_xlabel('Year',fontsize=14)
        axes_flat_spring[j].set_ylabel(interest_variable[j],fontsize=14)
        axes_flat_spring[j].set_xticks(x_ticks)
        axes_flat_spring[j].set_xticklabels(sparse_labels,rotation=70)
        if 'DOY' in interest_variable[j]:
            axes_flat_spring[j].axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
    fig_spring.suptitle(f"{titles[i]} Spring Bloom Metrics",fontsize=20)
    filename = f"{region_acro[i]}_Spring_Bloom_Metrics_scatter"
    plt.tight_layout()
    plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
    plt.close(fig_spring)

#### Georges Bank Fall and Spring Metrics

In [63]:
region_acro = 'GB'
data = bloom_GB
color_options = 'darkorange'
interest_variable= ['Start_DOY', 'Peak_DOY', 'End_DOY']
variable_title= ['Initiation Day of Year', 'Peak Day of Year', 'Termination Day of Year']
titles = 'Georges Bank'
x_ticks = [1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]
x_labels = ['1998','1999','2000','2001','2002','2003','2004','2005','2006','2007','2008','2009','2010','2011','2012','2013','2014','2015','2016','2017','2018','2019','2020','2021','2022','2023','2024','2025']
sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(x_labels)]
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15,9))
axes_flat = axes.flatten()
for j in range(len(interest_variable)):
    fall_x, fall_y, spring_x, spring_y = scatter_plot_data(dataset=data,interest_variable=interest_variable[j])
    ax_fall = axes_flat[j]
    sns.regplot(
        x=fall_x,
        y=fall_y,
        ax=ax_fall,
        color='goldenrod'
    )
    fall_x = np.array(fall_x)
    fall_y = np.array(fall_y)
    mask = ~np.isnan(fall_x) & ~np.isnan(fall_y)
    clean_fall_x = fall_x[mask]
    clean_fall_y = fall_y[mask]
    slope,intercept,r_value,p_value,std_err = stats.linregress(clean_fall_x,clean_fall_y)
    r_square = r_value**2
    stats_text = f"p-value = {p_value:.4f}"
    if p_value <=0.0559:
        ax_fall.text(
            0.05,0.95,
            stats_text,
            transform=ax_fall.transAxes,
            fontsize=15,
            color='red',
            verticalalignment='top'
        )
    else:
        ax_fall.text(
            0.05,0.95,
            stats_text,
            transform=ax_fall.transAxes,
            fontsize=15,
            color='black',
            verticalalignment='top'
        )
    ax_fall.set_title("Fall Bloom " + variable_title[j],fontsize=18)
    ax_fall.set_xlabel('Year',fontsize=14)
    ax_fall.set_ylabel(interest_variable[j],fontsize=14)
    ax_fall.set_xticks(x_ticks)
    ax_fall.set_xticklabels(sparse_labels,rotation=70)
    ax_fall.set_ylim(220,440)
    if 'DOY' in interest_variable[j]:
        ax_fall.axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)

    #Spring
    ax_spring = axes_flat[j+3]
    sns.regplot(
    x=spring_x,
    y=spring_y,
    ax=ax_spring,
    color='forestgreen'
    )
    spring_x = np.array(spring_x)
    spring_y = np.array(spring_y)
    mask = ~np.isnan(spring_x) & ~np.isnan(spring_y)
    clean_spring_x = spring_x[mask]
    clean_spring_y = spring_y[mask]
    slope,intercept,r_value,p_value,std_err = stats.linregress(clean_spring_x,clean_spring_y)
    r_square = r_value**2
    stats_text = f"p-value = {p_value:.4f}"
    if p_value <=0.0559:
        ax_spring.text(
            0.05,0.95,
            stats_text,
            transform=ax_spring.transAxes,
            fontsize=15,
            color='red',
            verticalalignment='top'
        )
    else:
        ax_spring.text(
            0.05,0.95,
            stats_text,
            transform=ax_spring.transAxes,
            fontsize=15,
            color='black',
            verticalalignment='top'
        )
    ax_spring.set_title("Spring Bloom " + variable_title[j],fontsize=18)
    ax_spring.set_xlabel('Year',fontsize=14)
    ax_spring.set_ylabel(interest_variable[j],fontsize=15)
    ax_spring.set_xticks(x_ticks)
    ax_spring.set_xticklabels(sparse_labels,rotation=70)
    ax_spring.set_ylim(25,175)
    if 'DOY' in interest_variable[j]:
        ax_spring.axhline(y=365, color='gray', linestyle='--', linewidth = 1, alpha=0.5)
fig.suptitle(f"{titles} Fall and Spring Bloom Metrics",fontsize=20)
filename = f"{region_acro}_Fall_Spring_Bloom_Metrics_scatter"
plt.tight_layout()
plt.savefig(rf'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Regional_scatter_plots\{filename}.png',dpi=300,bbox_inches='tight')
plt.close(fig)

### Integrated Chlorophyll Bar Charts

In [ ]:
titles = ['Middle Atlantic Bight South','Middle Atlantic Bight North','Georges Bank','Gulf of Maine West','Gulf of Maine East']
MABS_pivot = bloom_MABS.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
MABN_pivot = bloom_MABN.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GB_pivot = bloom_GB.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GOMW_pivot = bloom_GOMW.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GOME_pivot = bloom_GOME.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
my_colors = ['darkorange','darkorchid','deeppink','mediumblue']
total_lists = [summary_MABS,summary_MABN,summary_GB,summary_GOMW,summary_GOME]
pivot_lists = [MABS_pivot,MABN_pivot,GB_pivot,GOMW_pivot,GOME_pivot]
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(14,8),sharey=True, sharex=True)
axes_flat = axes.flatten()
#x_labels = ['1997/98','1998/99','1999/2000','2000/01','2001/02','2002/03','2003/04','2004/05','2005/06','2006/07','2007/08','2008/09','2009/10','2010/11','2011/12','2012/13','2013/14','2014/15','2015/16','2016/17','2017/18','2018/19','2019/20','2020/21','2021/22','2022/23','2023/24','2024/25','2025/26']
#sparse_labels = [label if idx % 4 == 0 else "" for idx, label in enumerate(x_labels)]
all_years = list(range(1997,2026))
for i in range(5):
    ax = axes_flat[i]
    df_summary = total_lists[i].copy()
    if 'Year' in df_summary.columns:
        df_summary = df_summary.set_index('Year')
    df_summary.index = df_summary.index.astype(int)
    clean_total_series = df_summary['Biological_total_integrated'].reindex(all_years,fill_value=0)
    clean_pivot = pivot_lists[i].reindex(all_years, fill_value=0)
    clean_total_series.plot(kind='bar',width=0.8,ax=ax,color='mediumseagreen',label='Non-Bloom Chlorophyll')
    clean_pivot.plot(kind='bar',stacked=True,width=0.8,ax=ax,legend=False,color=my_colors)
    ax.set_xlabel('Bloom Year')
    ax.tick_params(labelleft=True)
    ax.tick_params(labelbottom=True)
    ax.set_xticks(range(len(all_years)))
    xticks = [str(int(x)) for x in range(1997,2026)]
    sparse_labels = [label if idx % 2 == 0 else "" for idx, label in enumerate(xticks)]
    ax.set_xticklabels(sparse_labels,rotation=80,fontsize=10)
    ax.set_ylim(0,950)
    ax.set_title(titles[i],fontsize=18)

axes_flat[5].axis('off')
axes_flat[1].tick_params(labelleft=True)
axes_flat[2].tick_params(labelleft=True)
axes_flat[1].tick_params(labelbottom=True)
axes_flat[2].tick_params(labelbottom=True)
axes_flat[0].tick_params(labelbottom=True)
handles,labels = axes_flat[0].get_legend_handles_labels()
axes_flat[5].legend(handles,labels,title='Bloom Classification',title_fontsize=16,fontsize=15,loc='center')
plt.suptitle("Regional Integrated Chlorophyll",fontsize=20)
fig.supylabel("Integrated Chlorophyll Concentration ($mg/m^3 *days$)")
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\region_int_chl_bio_year.png',dpi=300,bbox_inches='tight')

In [ ]:
titles = ['Middle Atlantic Bight South','Georges Bank','Gulf of Maine East']
MABS_pivot = bloom_MABS.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GB_pivot = bloom_GB.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
GOME_pivot = bloom_GOME.pivot_table(
    values='Bloom_Integrated_Chlorophyll',
    index='Year',
    columns='Bloom_classification',
    aggfunc='sum',
    fill_value=0
)
my_colors = ['darkorange','darkorchid','deeppink','mediumblue']
total_lists = [summary_MABS,summary_GB,summary_GOME]
pivot_lists = [MABS_pivot,GB_pivot,GOME_pivot]
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(15,5),sharey=True)
axes_flat = axes.flatten()
#x_labels = ['1997/98','1998/99','1999/2000','2000/01','2001/02','2002/03','2003/04','2004/05','2005/06','2006/07','2007/08','2008/09','2009/10','2010/11','2011/12','2012/13','2013/14','2014/15','2015/16','2016/17','2017/18','2018/19','2019/20','2020/21','2021/22','2022/23','2023/24','2024/25','2025/26']
#sparse_labels = [label if idx % 4 == 0 else "" for idx, label in enumerate(x_labels)]
all_years = list(range(1997,2026))
for i in range(3):
    ax = axes_flat[i]
    df_summary = total_lists[i].copy()
    if 'Year' in df_summary.columns:
        df_summary = df_summary.set_index('Year')
    df_summary.index = df_summary.index.astype(int)
    clean_total_series = df_summary['Bloom_year_integrated_chl'].reindex(all_years,fill_value=0)
    clean_pivot = pivot_lists[i].reindex(all_years, fill_value=0)
    clean_total_series.plot(kind='bar',width=0.8,ax=ax,color='mediumseagreen',label='Non-Bloom Chlorophyll')
    clean_pivot.plot(kind='bar',stacked=True,width=0.8,ax=ax,legend=False,color=my_colors)
    ax.set_xlabel('Bloom Year')
    ax.tick_params(labelleft=True)
    ax.tick_params(labelbottom=True)
    ax.set_xticks(range(len(all_years)))
    xticks = [str(int(x)) for x in range(1997,2026)]
    sparse_labels = [label if idx % 3 == 0 else "" for idx, label in enumerate(xticks)]
    ax.set_xticklabels(sparse_labels,rotation=80,fontsize=10)
    ax.set_ylim(0,950)
    ax.set_title(titles[i],fontsize=18)

axes_flat[1].tick_params(labelleft=True)
axes_flat[2].tick_params(labelleft=True)
axes_flat[1].tick_params(labelbottom=True)
axes_flat[2].tick_params(labelbottom=True)
axes_flat[0].tick_params(labelbottom=True)
plt.suptitle("Regional Integrated Chlorophyll",fontsize=20)
fig.supylabel("Integrated Chlorophyll Concentration ($mg/m^3 *days$)")
plt.tight_layout()
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\region_int_chl_limited.png',dpi=300,bbox_inches='tight')
handles,labels = axes_flat[0].get_legend_handles_labels()
fig_legend = plt.figure(figsize=(3,2))
ax_leg = fig_legend.add_subplot(111)

legend = ax_leg.legend(handles,labels,loc='center')
ax_leg.axis('off')
plt.savefig(r'C:\Users\grace.davis\Documents\GitHub\phytoplankton_Hollings_project\Figures\Annual_metrics\limited_bars_legend.png',dpi=300,bbox_inches='tight')